In [ ]:
# Run once on first open. Skip / comment out after the first successful run.
!pip3 install --quiet numpy pandas matplotlib seaborn plotly statsmodels scikit-learn


# Round 5 — Within-Group Combinatorial Search: **Drinks (Oxygen Shakes)**

This notebook is a focused, in-depth correlation / cointegration search **inside a single 5-product group** (Drinks (Oxygen Shakes)). It mirrors `combination_search.ipynb` but is restricted to one group, and adds within-group sections that are only meaningful when products are expected to share a strong common factor.

## Why a within-group notebook?

All 5 variants of the same item are very likely to share a single dominant factor (raw-material cost, demand for the category, etc.). Cross-group analysis is almost guaranteed to surface that factor as ‘structural’ correlation. The interesting question inside a group is the **opposite**:

> *After removing the common factor, do any pairs / baskets in this group still mean-revert?*

Edges of that form are pure relative-value trades inside the group, and they tend to be much cleaner than cross-group trades because the products share market microstructure (same ticks, same liquidity profile, same observers).

## Search ladder

| § | Search | What we find |
|---|---|---|
| §2 | Data load + raw price viz | sanity, regime shifts |
| §3 | Pearson + Spearman + Kendall — all 1↔1 | which products co-move (level + returns + ranks) |
| §4 | Rolling correlation stability | does the relationship hold across days? |
| §5 | Lead-lag cross-correlation | which product moves first |
| §6 | 1↔1 cointegration (all 10 pairs) | best raw pairs by EG / ADF |
| §7 | 1↔1 integer hedge ratios | clean ratios like `A − 2·B` |
| §8 | PCA factor decomposition | how many factors drive the group |
| §9 | Composite index + per-product residual | which variant has the best idiosyncratic mean reversion |
| §10 | 1↔many continuous basket K=1..4 | best real-weight basket per target |
| §11 | 1↔many integer basket K=1..4 | clean integer baskets |
| §12 | Pure integer linear combos k=2..5 | stationary baskets without a designated target |
| §13 | Dashboard + residual plots | one-line winners + visual triage |

All sections work on the same 5-product aligned panel, so this notebook should run end-to-end in seconds (not minutes).


## §1 — Setup, config, helpers


In [ ]:
import math
import time
import warnings
from pathlib import Path
from itertools import combinations, product as iproduct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import kendalltau
from sklearn.decomposition import PCA
from statsmodels.tsa.stattools import adfuller, coint
from IPython.display import display, Markdown

%matplotlib inline
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# ============ CONFIG ============
DATA_DIR  = Path("ROUND_5")
ROUND     = 5
GROUP_KEY = "oxygen_shakes"
GROUP_LABEL = "Drinks (Oxygen Shakes)"
OUT_DIR   = Path("combination_search_drinks_outputs")
OUT_DIR.mkdir(exist_ok=True)
# ================================

PRODUCTS = [
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",
]
POSITION_LIMIT = 10  # all Round-5 products

# --- Combinatorial knobs (tiny within a 5-product group, so we can be exhaustive) ---
K_MAX             = 4       # max basket size for §10/§11; 4 is exhaustive (target + 4 others = 5)
INT_COEF_MIN      = -5
INT_COEF_MAX      =  5
TOP_PER_TARGET_K  = 50
TOP_DISPLAY       = 40
SUBSAMPLE_FOR_ADF = 20000   # random-ish row subsample for ADF (None = all)
RANDOM_SEED       = 42
WINDOW_FAST       = 200     # rolling window for residual plots
WINDOW_ROLLING    = 5000    # rolling window for stability check
MAX_LAG_LEAD      = 50      # ±lags for lead-lag cross-correlation

print(f"Group: {GROUP_LABEL}  ({len(PRODUCTS)} products)")
for p in PRODUCTS:
    print("  -", p)


In [ ]:
# ─── stat helpers ───
def adf_p(series, subsample=None):
    """ADF p-value. Optionally subsample for speed."""
    if subsample is None:
        subsample = SUBSAMPLE_FOR_ADF
    s = pd.Series(series).dropna().values
    if len(s) < 30 or np.std(s) == 0:
        return np.nan
    if subsample and len(s) > subsample:
        idx = np.linspace(0, len(s) - 1, subsample).astype(int)
        s = s[idx]
    try:
        return float(adfuller(s, autolag="AIC")[1])
    except Exception:
        return np.nan


def half_life(series):
    """OU half-life in ticks. NaN if non-reverting."""
    s = pd.Series(series).dropna().values
    if len(s) < 50:
        return np.nan
    x_lag = s[:-1]
    dx    = np.diff(s)
    X     = np.column_stack([np.ones_like(x_lag), x_lag])
    try:
        b = np.linalg.lstsq(X, dx, rcond=None)[0][1]
    except Exception:
        return np.nan
    return float(np.log(2) / -b) if (np.isfinite(b) and b < 0) else np.nan


def fmt_basket(prods, coefs, max_terms=8):
    """Format a (signed) basket like '3*A − 2*B + C'."""
    parts = []
    for p, c in zip(prods, coefs):
        if abs(c) < 1e-9:
            continue
        sign = "+" if c > 0 else "−"
        mag  = abs(c)
        mag_s = f"{int(round(mag))}" if abs(mag - round(mag)) < 1e-6 else f"{mag:.3f}"
        if mag_s == "1":
            parts.append(f"{sign} {p}")
        else:
            parts.append(f"{sign} {mag_s}*{p}")
    if not parts:
        return "(zero)"
    out = " ".join(parts)
    if out.startswith("+ "):
        out = out[2:]
    return out[:200]


## §2 — Data load + aligned mid panel + raw viz

Load every `prices_round_5_day_*.csv`, restrict to the 5 products of this group, build the global tick index `t = day · 1e6 + timestamp`, ffill empty-book mids, and pivot to a wide aligned panel. Then build the Gram matrix `G = Xcᵀ Xc / T` once (5×5, trivial) which underpins every downstream search.


In [ ]:
import re

def load_round5(data_dir: Path, products: list) -> pd.DataFrame:
    files = sorted(data_dir.glob("prices_round_*_day_*.csv"))
    if not files:
        raise FileNotFoundError(f"No prices CSVs in {data_dir}")
    frames = []
    for f in files:
        df = pd.read_csv(f, sep=";")
        if "day" not in df.columns:
            m = re.search(r"day_(-?\d+)", f.stem)
            if m:
                df["day"] = int(m.group(1))
        frames.append(df[df["product"].isin(products)])
    prices = pd.concat(frames, ignore_index=True)
    prices["t"] = prices["day"].astype(int) * 1_000_000 + prices["timestamp"].astype(int)
    prices.loc[prices["mid_price"] == 0, "mid_price"] = np.nan
    prices["mid_price"] = prices.groupby("product")["mid_price"].ffill()
    return prices


prices = load_round5(DATA_DIR, PRODUCTS)
print(f"Loaded {len(prices):,} rows across days {sorted(prices['day'].unique())}")
missing = [p for p in PRODUCTS if p not in set(prices['product'])]
if missing:
    print(f"⚠ Missing from data: {missing}")

mid = prices.pivot_table(index="t", columns="product", values="mid_price", aggfunc="last")
present = [p for p in PRODUCTS if p in mid.columns]
mid = mid[present].sort_index()
mid_aligned = mid.dropna()
T, P = mid_aligned.shape
print(f"\nAligned panel: T = {T:,} ticks × P = {P} products")

# Numpy backing arrays + Gram matrix
X       = mid_aligned.values.astype(np.float64)
columns = list(mid_aligned.columns)
mean_X  = X.mean(axis=0)
Xc      = X - mean_X
G       = (Xc.T @ Xc) / T
var     = np.diag(G).copy()
std     = np.sqrt(var)
rets    = mid_aligned.diff().iloc[1:]

summary = pd.DataFrame({
    "mean":    mid_aligned.mean(),
    "std":     std,
    "min":     mid_aligned.min(),
    "max":     mid_aligned.max(),
    "ret_std": rets.std(),
})
print("\n=== Per-product summary ===")
display(summary.round(3))


In [ ]:
# Raw mid-price plot — separate panels because absolute scales may differ
fig = make_subplots(rows=len(columns), cols=1, shared_xaxes=True,
                    vertical_spacing=0.02,
                    subplot_titles=columns)
for i, p in enumerate(columns):
    fig.add_trace(go.Scatter(y=mid_aligned[p].values, name=p, line=dict(width=0.7)),
                  row=i+1, col=1)
fig.update_layout(height=180 * len(columns), title=f"{GROUP_LABEL}: raw mid prices",
                  showlegend=False, hovermode="x unified")
fig.show()

# Z-scored overlay so co-movement is visually obvious
z = (mid_aligned - mid_aligned.mean()) / mid_aligned.std()
fig2 = go.Figure()
for p in columns:
    fig2.add_trace(go.Scatter(y=z[p].values, name=p, line=dict(width=0.8)))
fig2.update_layout(height=420, title=f"{GROUP_LABEL}: z-scored mids overlay",
                   hovermode="x unified")
fig2.show()


## §3 — Pearson + Spearman + Kendall correlations

Three rankings of the same C(5,2) = 10 pairs:

1. **Pearson (level)** — raw mid co-movement. High here alone is suspect (shared drift).
2. **Pearson (returns / first-difference)** — tick-by-tick co-movement. The honest signal.
3. **Spearman / Kendall (returns)** — rank-based, robust to outliers. Should agree with Pearson(returns) for tightly-traded products; large divergence flags a non-linear relationship or a few influential ticks.

We then compute a joint score `√(|pearson_level| · |pearson_ret|)` to surface pairs where **both** are high — the strongest pair-trade candidates.


In [ ]:
corr_level    = mid_aligned.corr(method="pearson")
corr_ret      = rets.corr(method="pearson")
corr_spearman = rets.corr(method="spearman")

# Kendall is O(n²) per pair so subsample for speed
kend_subsample = min(len(rets), 5000)
rets_kend = rets.sample(n=kend_subsample, random_state=RANDOM_SEED).sort_index() if len(rets) > kend_subsample else rets
corr_kendall = rets_kend.corr(method="kendall")


def pair_long(C: pd.DataFrame, val_name: str) -> pd.DataFrame:
    rows = []
    cols = list(C.columns)
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if i >= j:
                continue
            rows.append({"a": a, "b": b, val_name: C.iat[i, j]})
    return pd.DataFrame(rows)


p_level = pair_long(corr_level,    "pearson_level")
p_ret   = pair_long(corr_ret,      "pearson_ret")
p_spear = pair_long(corr_spearman, "spearman_ret")
p_kend  = pair_long(corr_kendall,  "kendall_ret")

both = p_level.merge(p_ret, on=["a", "b"]) \
              .merge(p_spear, on=["a", "b"]) \
              .merge(p_kend,  on=["a", "b"])
both["abs_level"]   = both["pearson_level"].abs()
both["abs_ret"]     = both["pearson_ret"].abs()
both["joint_score"] = np.sqrt(both["abs_level"] * both["abs_ret"])
both["par_minus_spear"] = (both["pearson_ret"] - both["spearman_ret"]).abs()  # non-linearity flag

both.to_csv(OUT_DIR / "pair_correlations.csv", index=False)

print("=== All 10 pairs sorted by joint score ===")
display(both.sort_values("joint_score", ascending=False).round(4))

print("\n=== Pairs where Pearson(ret) and Spearman(ret) DISAGREE most ===")
display(both.sort_values("par_minus_spear", ascending=False).head(5).round(4))

# Heatmaps
fig, axs = plt.subplots(2, 2, figsize=(13, 10))
for ax, C, title in [
    (axs[0, 0], corr_level,    "Pearson — level"),
    (axs[0, 1], corr_ret,      "Pearson — returns"),
    (axs[1, 0], corr_spearman, "Spearman — returns"),
    (axs[1, 1], corr_kendall,  f"Kendall — returns (n={kend_subsample})"),
]:
    sns.heatmap(C, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
                annot=True, fmt=".3f", cbar_kws={"shrink": 0.7}, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", labelsize=7, rotation=45)
    ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()


In [ ]:
# Pairwise scatter matrix on returns — visual sanity check for non-linear / heavy-tailed structure
sample_n = min(len(rets), 3000)
rets_s = rets.sample(n=sample_n, random_state=RANDOM_SEED) if len(rets) > sample_n else rets
sns.set(style="white")
g = sns.pairplot(rets_s, plot_kws=dict(s=4, alpha=0.3), diag_kind="kde", height=1.7)
g.fig.suptitle(f"{GROUP_LABEL}: pairwise return scatter (n={sample_n})", y=1.02, fontsize=11)
plt.show()
sns.set_theme(style="whitegrid", font_scale=0.9)


## §4 — Rolling correlation stability

Single-number Pearson averages a relationship over the whole panel. A pair can have decent overall correlation while flipping sign within the window — useless for trading.

We compute rolling Pearson(returns) over a window of `WINDOW_ROLLING` ticks for every pair and look at:

- The **time-series** of rolling correlation (eyeball stability + regime changes)
- The **range** [min, max] across the window — pairs with a tight range are reliable
- The **fraction of windows above 0.5** (or below −0.5) — a robust ‘consistency’ score


In [ ]:
win = min(WINDOW_ROLLING, max(500, T // 20))
rolling_rows = []
rolling_series = {}
for a, b in combinations(columns, 2):
    rc = rets[a].rolling(win).corr(rets[b]).dropna()
    if len(rc) == 0:
        continue
    rolling_series[f"{a}|{b}"] = rc
    rolling_rows.append({
        "a": a, "b": b,
        "mean":   float(rc.mean()),
        "std":    float(rc.std()),
        "min":    float(rc.min()),
        "max":    float(rc.max()),
        "frac_above_0.5": float((rc.abs() > 0.5).mean()),
        "frac_above_0.3": float((rc.abs() > 0.3).mean()),
    })
rolling_summary = pd.DataFrame(rolling_rows).sort_values("frac_above_0.5", ascending=False)
rolling_summary.to_csv(OUT_DIR / "rolling_correlation_summary.csv", index=False)

print(f"Rolling Pearson(returns) summary, window={win} ticks:")
display(rolling_summary.round(3))

# Plot all 10 rolling correlations
fig = go.Figure()
for k, rc in rolling_series.items():
    fig.add_trace(go.Scatter(y=rc.values, name=k, line=dict(width=0.8)))
fig.add_hline(y=0, line_dash="dot", line_color="black")
fig.update_layout(height=480, title=f"{GROUP_LABEL}: rolling Pearson(returns), window={win}",
                  hovermode="x unified")
fig.show()


## §5 — Lead-lag cross-correlation

For each ordered pair `(a, b)`, compute `corr(rets[a], rets[b].shift(L))` for L ∈ {−MAX_LAG_LEAD … MAX_LAG_LEAD}. The lag with peak |correlation| says which product **leads** the other (positive L: `a` leads `b`).

Within a tight group, ‘leadership’ is usually informational — one variant is the price-discovery instrument and the rest follow. That's a tradeable structural feature: if A leads B and the contemporaneous spread blows out, the next-tick movement in B may be predictable.


In [ ]:
lags = list(range(-MAX_LAG_LEAD, MAX_LAG_LEAD + 1))
leadlag_curves = {}
leadlag_rows = []
for a, b in combinations(columns, 2):
    ra = rets[a]
    rb = rets[b]
    cs = []
    for L in lags:
        if L >= 0:
            c = ra.iloc[L:].reset_index(drop=True).corr(rb.iloc[:len(rb) - L].reset_index(drop=True))
        else:
            c = ra.iloc[:len(ra) + L].reset_index(drop=True).corr(rb.iloc[-L:].reset_index(drop=True))
        cs.append(c)
    cs = np.asarray(cs)
    leadlag_curves[f"{a}|{b}"] = cs
    best_idx = int(np.nanargmax(np.abs(cs)))
    best_lag = lags[best_idx]
    leadlag_rows.append({
        "a": a, "b": b,
        "corr_at_0":   float(cs[lags.index(0)]),
        "peak_corr":   float(cs[best_idx]),
        "peak_abs":    float(abs(cs[best_idx])),
        "peak_lag":    int(best_lag),
        "interpretation": (
            "contemporaneous" if best_lag == 0
            else (f"{a} leads {b} by {best_lag}" if best_lag > 0 else f"{b} leads {a} by {-best_lag}")
        ),
    })
leadlag_summary = pd.DataFrame(leadlag_rows).sort_values("peak_abs", ascending=False)
leadlag_summary.to_csv(OUT_DIR / "leadlag_summary.csv", index=False)

print("=== Lead-lag summary (returns), peak |corr| within ±%d ticks ===" % MAX_LAG_LEAD)
display(leadlag_summary.round(4))

# Plot the curves
fig = go.Figure()
for k, cs in leadlag_curves.items():
    fig.add_trace(go.Scatter(x=lags, y=cs, name=k, mode="lines"))
fig.add_vline(x=0, line_dash="dot", line_color="black")
fig.update_layout(height=440,
                  title=f"{GROUP_LABEL}: lead-lag cross-correlation of returns (positive lag: a leads b)",
                  xaxis_title="lag L (ticks)", yaxis_title="corr(rets[a], rets[b].shift(L))",
                  hovermode="x unified")
fig.show()


## §6 — Pairwise cointegration (all 10 pairs)

For every pair `(a, b)`:

1. OLS hedge ratio `h = cov(a, b) / var(b)` (closed form via Gram matrix).
2. Spread `s = a − h·b`, residual std ratio = `√(1 − corr²)`.
3. **Engle-Granger** cointegration p-value (`coint`) — uses the right critical values for an OLS-fitted spread.
4. ADF p-value on the spread.
5. OU half-life of the spread.

Within 5 products this is 10 pairs — we run the full test on every one, no two-stage shortlist.


In [ ]:
COINT_MAXLAG = 5

def coint_p_fast(xi, xj):
    try:
        return float(coint(xi, xj, trend="c", maxlag=COINT_MAXLAG, autolag=None)[1])
    except Exception:
        return np.nan


def adf_p_fast(spread):
    s = pd.Series(spread).dropna().values
    if SUBSAMPLE_FOR_ADF and len(s) > SUBSAMPLE_FOR_ADF:
        s = s[np.linspace(0, len(s) - 1, SUBSAMPLE_FOR_ADF).astype(int)]
    if len(s) < 30 or np.std(s) == 0:
        return np.nan
    try:
        return float(adfuller(s, maxlag=COINT_MAXLAG, autolag=None)[1])
    except Exception:
        return np.nan


rows = []
for i, j in combinations(range(P), 2):
    if var[j] == 0 or var[i] == 0:
        continue
    h        = G[i, j] / var[j]
    rvar     = var[i] - h * G[i, j]
    lvl_corr = G[i, j] / np.sqrt(var[i] * var[j])
    spread   = X[:, i] - h * X[:, j]
    rows.append({
        "a": columns[i], "b": columns[j],
        "hedge_ratio":     float(h),
        "level_corr":      float(lvl_corr),
        "resid_std_ratio": float(np.sqrt(max(rvar, 0) / var[i])),
        "eg_p":            coint_p_fast(X[:, i], X[:, j]),
        "adf_p":           adf_p_fast(spread),
        "half_life":       half_life(spread),
    })
pair_table = pd.DataFrame(rows).sort_values("eg_p").reset_index(drop=True)
pair_table.to_csv(OUT_DIR / "pair_cointegration.csv", index=False)

print("=== All pairs by EG p-value ===")
display(pair_table.round(4))


## §7 — Pairwise integer hedge ratios

Real-valued hedge ratios are messy in practice (you can't trade 1.7 units). Search the integer grid `[INT_COEF_MIN..INT_COEF_MAX] \ {0}` per pair and pick the integer that minimises spread variance:

`var(a − k·b) = var(a) − 2·k·cov(a,b) + k²·var(b)`

(closed form via Gram matrix). Then run ADF + half-life on the integer spread.

**Position-limit reminder**: limit is **10**, so a pair `a − k·b` traded at one unit means `±10 a, ∓10·k b` at full size. Prefer `|k| ≤ 3`.


In [ ]:
int_grid = [k for k in range(INT_COEF_MIN, INT_COEF_MAX + 1) if k != 0]
int_rows = []
for _, r in pair_table.iterrows():
    a, b = r["a"], r["b"]
    i, j = columns.index(a), columns.index(b)
    best_k, best_v = None, np.inf
    for k in int_grid:
        v = var[i] - 2 * k * G[i, j] + k ** 2 * var[j]
        if v < best_v:
            best_v, best_k = v, k
    spread = X[:, i] - best_k * X[:, j]
    int_rows.append({
        "a": a, "b": b,
        "ols_h":               r["hedge_ratio"],
        "int_h":               int(best_k),
        "ols_resid_std_ratio": r["resid_std_ratio"],
        "int_resid_std_ratio": float(np.sqrt(max(best_v, 0) / var[i])),
        "int_resid_inflation": float(np.sqrt(max(best_v, 0) / var[i]) / max(r["resid_std_ratio"], 1e-9)),
        "int_adf_p":           adf_p(spread),
        "int_half_life":       half_life(spread),
    })
int_pair_table = pd.DataFrame(int_rows).sort_values("int_adf_p").reset_index(drop=True)
int_pair_table.to_csv(OUT_DIR / "pair_integer_hedge.csv", index=False)

print("=== Integer-hedge pairs by ADF p ===")
display(int_pair_table.round(4))

print("\n=== Pairs where rounding is nearly free (inflation < 1.05) ===")
display(int_pair_table.query("int_resid_inflation < 1.05").sort_values("int_adf_p").round(4))


## §8 — PCA factor decomposition

Run PCA on the **return panel** (centered, *not* scaled — same units, similar volatility). For 5 highly correlated products we expect:

- **PC1** carries the dominant common factor (group-wide drift). Loading sign and magnitude tells us which variant moves with the group most.
- **PC2..PC5** are the idiosyncratic axes. Each PC's residuals are an orthogonal candidate spread; if `PC2` (say) is mean-reverting, we have a tradeable basket whose weights are the PC2 loadings.

We test every PC residual for ADF / half-life. PC1 should *not* be mean-reverting (it's the trend). PC2..PC5 might.


In [ ]:
pca = PCA(n_components=P)
ret_arr = rets.values
ret_centered = ret_arr - ret_arr.mean(axis=0)
scores = pca.fit_transform(ret_centered)  # (T-1, P)
loadings = pca.components_  # (P, P) — rows are PCs, columns are products

explained = pd.DataFrame({
    "pc": [f"PC{i+1}" for i in range(P)],
    "explained_var_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
})
print("=== Variance explained per PC ===")
display(explained.round(4))

load_df = pd.DataFrame(loadings,
                       index=[f"PC{i+1}" for i in range(P)],
                       columns=columns)
print("\n=== Loadings (rows: PCs, cols: products) ===")
display(load_df.round(3))

# Reconstruct PC residuals at the LEVEL (cumulative sum of returns gives a level-like series)
pc_levels = pd.DataFrame(np.cumsum(scores, axis=0),
                         columns=[f"PC{i+1}" for i in range(P)])

pc_rows = []
for i in range(P):
    name   = f"PC{i+1}"
    score  = scores[:, i]
    level  = pc_levels[name].values
    pc_rows.append({
        "pc":                  name,
        "explained_var_ratio": float(pca.explained_variance_ratio_[i]),
        "loadings":            load_df.loc[name].round(3).to_dict(),
        "adf_p_score":         adf_p(score),         # returns-space (should be small for stationary noise)
        "adf_p_level":         adf_p(level),         # cumulative — small means stationary, NOT random walk
        "half_life_level":     half_life(level),
    })
pc_table = pd.DataFrame(pc_rows)
pc_table.to_csv(OUT_DIR / "pca_residuals.csv", index=False)
print("\n=== PC residual stationarity ===")
display(pc_table)

# Plot the level-space PC trajectories
fig = make_subplots(rows=P, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[f"PC{i+1}  (var={pca.explained_variance_ratio_[i]:.2%})"
                                    for i in range(P)])
for i in range(P):
    fig.add_trace(go.Scatter(y=pc_levels.iloc[:, i].values, line=dict(width=0.8),
                             name=f"PC{i+1}"), row=i+1, col=1)
fig.update_layout(height=180 * P, showlegend=False, hovermode="x unified",
                  title=f"{GROUP_LABEL}: PC level trajectories (cumulative scores)")
fig.show()

# Loadings heatmap
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(load_df, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            cbar_kws={"label": "loading"}, ax=ax)
ax.set_title(f"{GROUP_LABEL}: PCA loadings")
plt.tight_layout(); plt.show()


## §9 — Composite index + per-product idiosyncratic residual

Build the group composite index as the **z-mean** of the 5 mids (same definition as §7 of the master notebook). For each product `p`, the residual `p − OLS(p ~ index)` is its idiosyncratic component. If that residual is mean-reverting, you can trade `p` against the index — and the basket is **automatically integer-friendly** because the index weight is uniform across the other 4 products.

We also report the OLS β of each product on the index: a high β means the product is the most responsive to the common factor, which is sometimes the cleanest pair-trade leg.


In [ ]:
z = (mid_aligned - mid_aligned.mean()) / mid_aligned.std(ddof=0).replace(0, np.nan)
z = z.dropna(axis=1)
index_series = z.mean(axis=1)
index_series.name = "GROUP_INDEX"
index_series.to_csv(OUT_DIR / "group_index.csv")

idx_rows = []
for p in columns:
    s = mid_aligned[p]
    # OLS p ~ a + b·index
    A = np.column_stack([np.ones_like(index_series.values), index_series.values])
    coef, *_ = np.linalg.lstsq(A, s.values, rcond=None)
    a_int, b_idx = coef
    fitted = A @ coef
    resid  = s.values - fitted
    idx_rows.append({
        "product":         p,
        "intercept":       float(a_int),
        "beta_to_index":   float(b_idx),
        "resid_std":       float(np.std(resid)),
        "r_squared":       float(1 - np.var(resid) / np.var(s.values)),
        "adf_p":           adf_p(resid),
        "half_life":       half_life(resid),
    })
idx_table = pd.DataFrame(idx_rows).sort_values("adf_p")
idx_table.to_csv(OUT_DIR / "product_vs_index.csv", index=False)
print("=== Each product regressed on the group composite index ===")
display(idx_table.round(4))

# Residual plots
fig = make_subplots(rows=P, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[f"{p} − OLS·index   (β={r['beta_to_index']:.3f}, ADF p={r['adf_p']:.3g})"
                                    for p, r in zip(columns, [idx_table[idx_table['product']==p].iloc[0] for p in columns])])
for i, p in enumerate(columns):
    s = mid_aligned[p]
    A = np.column_stack([np.ones_like(index_series.values), index_series.values])
    coef, *_ = np.linalg.lstsq(A, s.values, rcond=None)
    resid = s.values - A @ coef
    fig.add_trace(go.Scatter(y=resid, line=dict(width=0.7), name=p), row=i+1, col=1)
    fig.add_hline(y=0, line_dash="dot", line_color="black", row=i+1, col=1)
fig.update_layout(height=180 * P, showlegend=False, hovermode="x unified",
                  title=f"{GROUP_LABEL}: idiosyncratic residual = product − OLS(product ~ index)")
fig.show()


## §10 — 1↔many continuous basket search (K = 1..4)

For each product as **target**, exhaustively try every subset of size `k = 1..4` from the other 4 products as a hedge basket. Within a 5-product group this is tiny:

- K=1: 5 × 4 = 20 fits
- K=2: 5 × C(4,2) = 30 fits
- K=3: 5 × C(4,3) = 20 fits
- K=4: 5 × C(4,4) = 5 fits

Total = 75 fits. Every one runs ADF + half-life (no shortlist needed). The Gram trick keeps each fit O(k³).


In [ ]:
def basket_resid_var(target_idx, S):
    Gss = G[np.ix_(S, S)]
    Gst = G[S, target_idx]
    try:
        beta = np.linalg.solve(Gss, Gst)
    except np.linalg.LinAlgError:
        return np.inf, None
    rvar = var[target_idx] - beta @ Gst
    if rvar < -1e-9:
        return np.inf, None
    return max(rvar, 0.0), beta


def reconstruct_residual(target_idx, S, beta):
    return Xc[:, target_idx] - Xc[:, S] @ beta


basket_results = {}
for k in range(1, K_MAX + 1):
    rows = []
    for ti in range(P):
        cands = [j for j in range(P) if j != ti]
        for combo in combinations(cands, k):
            S = list(combo)
            rvar, beta = basket_resid_var(ti, S)
            if not np.isfinite(rvar) or beta is None:
                continue
            target = columns[ti]
            basket = [columns[s] for s in S]
            resid  = reconstruct_residual(ti, S, beta)
            rows.append({
                "target":     target,
                "k":          k,
                "basket":     basket,
                "ols_beta":   beta.tolist(),
                "resid_var":  float(rvar),
                "r_squared":  float(1 - rvar / var[ti]) if var[ti] > 0 else np.nan,
                "adf_p":      adf_p(resid),
                "half_life":  half_life(resid),
                "S_idx":      S,
                "target_idx": ti,
            })
    df = pd.DataFrame(rows).sort_values("adf_p").reset_index(drop=True)
    df["basket_str"] = df.apply(
        lambda r: fmt_basket([r["target"]] + r["basket"], [1.0] + list(-np.array(r["ols_beta"]))),
        axis=1)
    basket_results[k] = df
    df.to_csv(OUT_DIR / f"basket_continuous_K{k}.csv", index=False)
    print(f"\n=== Top continuous baskets — K = {k} (sorted by ADF p) ===")
    display(df.head(TOP_DISPLAY)[
        ["target", "k", "basket_str", "r_squared", "adf_p", "half_life"]
    ].round(4))


## §11 — 1↔many integer-coefficient basket search

For every continuous basket above, find the closest integer vector that still minimises residual variance:

1. Round OLS β to nearest integer in `[INT_COEF_MIN, INT_COEF_MAX]`.
2. Search the 3ᵏ neighbourhood around the rounded vector.
3. For tiny baskets (k ≤ 2) also try the **full** integer grid.
4. Pick the integer vector minimising `cᵀ G[S,S] c − 2 cᵀ G[S,t] + G[t,t]`.

Then run ADF + half-life on the integer spread and report the **inflation ratio** `int_resid_std / ols_resid_std`. Inflation < 1.05 means rounding cost is essentially free.


In [ ]:
def integer_var(c_int, target_idx, S):
    c = np.asarray(c_int, dtype=float)
    return var[target_idx] - 2 * c @ G[S, target_idx] + c @ G[np.ix_(S, S)] @ c


def integer_search_local(beta, target_idx, S, radius=1):
    near = np.clip(np.round(beta).astype(int), INT_COEF_MIN, INT_COEF_MAX)
    best_c, best_v = None, np.inf
    deltas = list(range(-radius, radius + 1))
    for d in iproduct(deltas, repeat=len(beta)):
        c = np.clip(near + np.array(d), INT_COEF_MIN, INT_COEF_MAX)
        if (c == 0).all():
            continue
        v = integer_var(c, target_idx, S)
        if v < best_v:
            best_v, best_c = v, c.copy()
    return best_c, best_v


def integer_search_full(target_idx, S):
    best_c, best_v = None, np.inf
    rng = list(range(INT_COEF_MIN, INT_COEF_MAX + 1))
    for c in iproduct(rng, repeat=len(S)):
        c_arr = np.asarray(c, dtype=int)
        if (c_arr == 0).all():
            continue
        v = integer_var(c_arr, target_idx, S)
        if v < best_v:
            best_v, best_c = v, c_arr.copy()
    return best_c, best_v


def integer_baskets_per_K(df_top_continuous, full_grid_k_max=3):
    rows = []
    for _, row in df_top_continuous.iterrows():
        ti    = row["target_idx"]
        S     = row["S_idx"]
        beta  = np.asarray(row["ols_beta"])
        if len(S) <= full_grid_k_max:
            c_full, v_full = integer_search_full(ti, S)
        else:
            c_full, v_full = None, np.inf
        c_local, v_local   = integer_search_local(beta, ti, S, radius=1)
        if c_full is not None and v_full < v_local:
            c_int, v_int = c_full, v_full
        else:
            c_int, v_int = c_local, v_local
        if c_int is None:
            continue
        target = columns[ti]
        basket = [columns[s] for s in S]
        spread = X[:, ti] - X[:, S] @ c_int.astype(float)
        rows.append({
            "target":              target,
            "k":                   len(S),
            "basket":              basket,
            "ols_beta":            beta.tolist(),
            "int_coefs":           [int(c) for c in c_int],
            "int_basket_str":      fmt_basket([target] + basket, [1.0] + list(-c_int.astype(float))),
            "ols_resid_std_ratio": float(np.sqrt(max(row["resid_var"], 0) / var[ti])),
            "int_resid_std_ratio": float(np.sqrt(max(v_int, 0) / var[ti])),
            "int_resid_inflation": float(np.sqrt(max(v_int, 0) / max(row["resid_var"], 1e-12))),
            "max_abs_coef":        int(max(abs(c) for c in c_int)),
            "int_adf_p":           adf_p(spread),
            "int_half_life":       half_life(spread),
        })
    return pd.DataFrame(rows).sort_values("int_adf_p").reset_index(drop=True)


basket_int = {}
for k, df_cont in basket_results.items():
    df_int = integer_baskets_per_K(df_cont)
    basket_int[k] = df_int
    df_int.to_csv(OUT_DIR / f"basket_integer_K{k}.csv", index=False)
    print(f"\n=== Top integer baskets — K = {k} (sorted by ADF p) ===")
    display(df_int.head(TOP_DISPLAY)[
        ["target", "k", "int_basket_str", "max_abs_coef",
         "int_resid_std_ratio", "int_resid_inflation", "int_adf_p", "int_half_life"]
    ].round(4))

    cheap = df_int.query("int_resid_inflation < 1.05 and max_abs_coef <= 3")
    if len(cheap):
        print(f"\n  → Cheap integer baskets (inflation < 1.05, max|c| ≤ 3) for K = {k}:")
        display(cheap.head(TOP_DISPLAY)[
            ["target", "int_basket_str", "max_abs_coef",
             "int_resid_std_ratio", "int_adf_p", "int_half_life"]
        ].round(4))


## §12 — Pure integer linear combinations (no target, no hedge)

Find baskets `c₁·P₁ + c₂·P₂ + … + cₖ·Pₖ` (k ∈ {2,3,4,5}) that mean-revert without designating any product as ‘target’. This is the arb-trader's lens — *any* cheap integer combination of the order book that reverts.

Algorithm per subset `S`:

1. Smallest-eigenvalue eigenvector of `G[S,S]` → unit-norm direction of least variance.
2. Scale so `max |vᵢ| = INT_COEF_MAX`, round, clip.
3. Search the 3ᵏ neighbourhood around the rounded vector; pick min variance.
4. Drop trivial cases (all-zero, single non-zero coefficient).
5. Filter by **variance ratio** `c·G[S,S]·c / Σ cᵢ²·var(Pᵢ)` — small ratio = combination cancels much of individual variance, i.e. genuinely co-integrates.

Inside a 5-product group all 26 subsets across k ∈ {2..5} are exhaustive — no sampling, no shortcuts.


In [ ]:
def best_integer_combo(S):
    Gs = G[np.ix_(S, S)]
    eigvals, eigvecs = np.linalg.eigh(Gs)
    v = eigvecs[:, 0]
    if np.max(np.abs(v)) == 0:
        return None, np.inf, np.inf
    scale = INT_COEF_MAX / np.max(np.abs(v))
    near = np.clip(np.round(v * scale).astype(int), INT_COEF_MIN, INT_COEF_MAX)
    best_c, best_v = None, np.inf
    for d in iproduct([-1, 0, 1], repeat=len(S)):
        c = np.clip(near + np.array(d), INT_COEF_MIN, INT_COEF_MAX)
        if (c == 0).all():
            continue
        if (c != 0).sum() < 2:
            continue
        v_combo = c @ Gs @ c
        if v_combo < best_v:
            best_v, best_c = v_combo, c.copy()
    if best_c is None:
        return None, np.inf, np.inf
    norm_var = sum((best_c[i] ** 2) * var[S[i]] for i in range(len(S)))
    var_ratio = float(best_v / max(norm_var, 1e-12))
    return best_c, float(best_v), var_ratio


pure_rows = []
for k in range(2, P + 1):
    for S in combinations(range(P), k):
        S_l = list(S)
        c_int, v_combo, vr = best_integer_combo(S_l)
        if c_int is None:
            continue
        spread = X[:, S_l] @ c_int.astype(float)
        pure_rows.append({
            "k":                 k,
            "basket":            [columns[s] for s in S_l],
            "int_coefs":         [int(c) for c in c_int],
            "basket_str":        fmt_basket([columns[s] for s in S_l], list(c_int.astype(float))),
            "max_abs_coef":      int(max(abs(c) for c in c_int)),
            "combo_var":         float(v_combo),
            "var_ratio":         vr,
            "adf_p":             adf_p(spread),
            "half_life":         half_life(spread),
        })
pure_table = pd.DataFrame(pure_rows).sort_values(["adf_p", "var_ratio"]).reset_index(drop=True)
pure_table.to_csv(OUT_DIR / "pure_integer_combos.csv", index=False)

print("=== Pure integer combinations sorted by ADF p, then var_ratio ===")
display(pure_table.round(4))

for k in sorted(pure_table["k"].unique()):
    sub = pure_table.query("k == @k")
    print(f"\nBest pure k={k} combo: {sub.iloc[0]['basket_str']}  "
          f"(ADF p={sub.iloc[0]['adf_p']:.3g}, half-life={sub.iloc[0]['half_life']})")


## §13 — Dashboard + residual plots for the strongest results

Compile one-line winners from §6, §7, §9, §10, §11, §12 into a single ranked shortlist, then plot residual time-series with ±2σ rolling band and z-score for the top entries. A clean tradeable spread shows many round-trips through the band; a single big move that never reverts is selection bias from running thousands of fits.


In [ ]:
def plot_residual(name, series, window=WINDOW_FAST):
    s = pd.Series(series).dropna()
    rm = s.rolling(window).mean()
    rs = s.rolling(window).std()
    z  = (s - rm) / rs
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[f"{name} — residual + ±2σ", "z-score"])
    fig.add_trace(go.Scatter(y=(rm + 2*rs).values, line=dict(width=0), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(y=(rm - 2*rs).values, line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(255,165,0,0.15)", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(y=s.values, line=dict(width=0.7, color="steelblue"), name="resid"), row=1, col=1)
    fig.add_trace(go.Scatter(y=rm.values, line=dict(width=1.0, dash="dot", color="orange"),
                             name="rolling mean"), row=1, col=1)
    fig.add_trace(go.Scatter(y=z.values, line=dict(width=0.7, color="purple"), name="z"), row=2, col=1)
    fig.update_layout(height=480, hovermode="x unified", showlegend=False, title=name)
    fig.show()


dashboard = []

if len(pair_table):
    r = pair_table.dropna(subset=["eg_p"]).iloc[0]
    dashboard.append({"section": "§6 best OLS pair",
                      "recipe":  f"{r['a']} − {r['hedge_ratio']:.3f}·{r['b']}",
                      "p_value": r["eg_p"], "half_life": r["half_life"]})

if len(int_pair_table):
    r = int_pair_table.dropna(subset=["int_adf_p"]).iloc[0]
    sign = "−" if r["int_h"] > 0 else "+"
    dashboard.append({"section": "§7 best integer pair",
                      "recipe":  f"{r['a']} {sign} {abs(r['int_h'])}·{r['b']}",
                      "p_value": r["int_adf_p"], "half_life": r["int_half_life"]})

if len(idx_table):
    r = idx_table.dropna(subset=["adf_p"]).iloc[0]
    dashboard.append({"section": "§9 best product-vs-index residual",
                      "recipe":  f"{r['product']} − ({r['intercept']:.2f} + {r['beta_to_index']:.3f}·index)",
                      "p_value": r["adf_p"], "half_life": r["half_life"]})

for k, df in basket_results.items():
    df_ok = df.dropna(subset=["adf_p"])
    if len(df_ok):
        r = df_ok.iloc[0]
        dashboard.append({"section": f"§10 best continuous K={k}",
                          "recipe":  r["basket_str"],
                          "p_value": r["adf_p"], "half_life": r["half_life"]})

for k, df in basket_int.items():
    df_ok = df.dropna(subset=["int_adf_p"])
    if len(df_ok):
        r = df_ok.iloc[0]
        dashboard.append({"section": f"§11 best integer K={k}",
                          "recipe":  r["int_basket_str"],
                          "p_value": r["int_adf_p"], "half_life": r["int_half_life"]})

if len(pure_table):
    for k in sorted(pure_table["k"].unique()):
        sub = pure_table.query("k == @k").dropna(subset=["adf_p"])
        if len(sub):
            r = sub.iloc[0]
            dashboard.append({"section": f"§12 best pure integer k={k}",
                              "recipe":  r["basket_str"],
                              "p_value": r["adf_p"], "half_life": r["half_life"]})

dash_df = pd.DataFrame(dashboard).sort_values("p_value").reset_index(drop=True)
dash_df.to_csv(OUT_DIR / "dashboard.csv", index=False)
print("=== Dashboard — winners by section, sorted by p-value ===")
display(dash_df.round(5))


In [ ]:
# Plot the residual for the top 6 dashboard entries
TOP_PLOTS = 6
for _, row in dash_df.head(TOP_PLOTS).iterrows():
    name = f"{row['section']}: {row['recipe']}"
    # Reconstruct the residual
    section = row["section"]
    series = None
    if "§6" in section:
        r = pair_table.dropna(subset=["eg_p"]).iloc[0]
        series = X[:, columns.index(r['a'])] - r['hedge_ratio'] * X[:, columns.index(r['b'])]
    elif "§7" in section:
        r = int_pair_table.dropna(subset=["int_adf_p"]).iloc[0]
        series = X[:, columns.index(r['a'])] - r['int_h'] * X[:, columns.index(r['b'])]
    elif "§9" in section:
        r = idx_table.dropna(subset=["adf_p"]).iloc[0]
        s = mid_aligned[r['product']].values
        A_ = np.column_stack([np.ones_like(index_series.values), index_series.values])
        coef, *_ = np.linalg.lstsq(A_, s, rcond=None)
        series = s - A_ @ coef
    elif "§10" in section:
        k = int(section.split("K=")[1])
        r = basket_results[k].dropna(subset=["adf_p"]).iloc[0]
        series = reconstruct_residual(r['target_idx'], r['S_idx'], np.asarray(r['ols_beta']))
    elif "§11" in section:
        k = int(section.split("K=")[1])
        r = basket_int[k].dropna(subset=["int_adf_p"]).iloc[0]
        S_idx = [columns.index(b) for b in r['basket']]
        c_int = np.asarray(r['int_coefs'], dtype=float)
        series = X[:, columns.index(r['target'])] - X[:, S_idx] @ c_int
    elif "§12" in section:
        k = int(section.split("k=")[1])
        r = pure_table.query("k == @k").dropna(subset=["adf_p"]).iloc[0]
        S_idx = [columns.index(b) for b in r['basket']]
        c_int = np.asarray(r['int_coefs'], dtype=float)
        series = X[:, S_idx] @ c_int
    if series is not None:
        plot_residual(name, series, window=WINDOW_FAST)


## §14 — How to read these results

Every CSV is in `combination_search_drinks_outputs/`:

- `pair_correlations.csv` — Pearson + Spearman + Kendall for all 10 pairs.
- `rolling_correlation_summary.csv` — stability of return correlation per pair.
- `leadlag_summary.csv` — peak cross-correlation lag per pair.
- `pair_cointegration.csv` — all 10 pairs with EG / ADF / half-life.
- `pair_integer_hedge.csv` — same pairs after integer hedge fitting.
- `pca_residuals.csv` — variance explained + ADF on each PC's residual.
- `group_index.csv`, `product_vs_index.csv` — composite index + per-product residual stats.
- `basket_continuous_K{k}.csv` — top continuous baskets per K.
- `basket_integer_K{k}.csv` — top integer baskets per K (the main output).
- `pure_integer_combos.csv` — pure integer combinations from eigenvector search.
- `dashboard.csv` — one-line winners per section.

### Triage rules (within a tightly-correlated group)

1. **Distrust strong level-correlation alone.** Within a group everything will look highly correlated at the level. Look at `pearson_ret`, `int_adf_p`, and the residual plot.
2. **Filter `adf_p < 0.05`** — those are statistically mean-reverting after the search-over-many cost.
3. **Filter `half_life ∈ [50, 500]`** — fast enough to round-trip in a round, slow enough to clear costs.
4. **Filter `int_resid_inflation < 1.10`** in §11 — integer rounding cost should be small.
5. **Prefer `max_abs_coef ≤ 3`** — position limit is 10, so a max coef of 3 lets you trade up to ~3 units of the basket at full size on the dominant leg.
6. **Eyeball the residual plot.** A real edge has many round-trips through ±2σ. One big move that never reverts is selection bias.

### What's special about within-group results

- §8 (PCA) tells you whether the group is essentially one factor (PC1 dominates ⇒ the only edge is variant-level relative value) or genuinely multi-factor (multiple PCs explain meaningful variance ⇒ richer structure).
- §9 (product-vs-index) is the cleanest ‘structural’ trade in the group: each product against the average of its peers. If only one product has a stationary residual, that's a strong signal — the others all swim with the school.
- §11 / §12 baskets that beat §7's integer pairs justify the extra complexity. If they don't, stick with the pair.

### Re-run knobs

- Tighten `INT_COEF_MAX` to 3 for stricter ‘cheap’ baskets.
- Widen `MAX_LAG_LEAD` if §5 shows the peak at the boundary (means longer lead is plausible).
- Set `SUBSAMPLE_FOR_ADF=None` for full-precision ADF on the final shortlist.


# Part II — Niche pivot: beyond direct correlation

Sections §1-§14 above are **direct-correlation** machinery: Pearson, Spearman, cointegration, basket regression, PCA. Inside a 5-product group those almost always surface the dominant common factor (a single shared drift), which dominates everything else.

The 5 oxygen-shake variants almost certainly share one common factor. The interesting question is **what's left after that factor is gone** — the niche relationships that direct correlation misses entirely:

| §  | What we test | Why direct correlation can't see it |
|----|---|---|
| §15 | Microstructure feature panel (spread, depth, OBI, microprice) | Mid throws away order-book information |
| §16 | Distance correlation + mutual information | Pearson sees only linear dependence |
| §17 | Tail / extreme co-movement (asymmetric, lagged) | Pearson averages the *body* of the distribution |
| §18 | Volatility cross-correlation (`|ret|`, `ret²`, rolling vol) | Two products can have Pearson(ret)=0 but co-volatility |
| §19 | Sign-asymmetric lead-lag + Granger F-tests | Symmetric corr conflates up-leads-up with down-leads-down |
| §20 | Cross-product OBI → forward return | Quote imbalance leaks before mid moves |
| §21 | Co-jump synchronization | Jumps are rare events; their co-incidence is hidden under the average |
| §22 | Signed trade-flow cross-effects | Executed flow ≠ resting quotes; carries committed demand |
| §23 | Cross-spectral coherence (frequency domain) | Time-domain corr aggregates over all frequencies |
| §24 | Niche dashboard | Roll up the strongest non-direct-correlation findings |

All sections work on the same aligned 5-product panel built in §2, plus a **microstructure feature panel** built in §15 (so §15 must run before §17-§22). Outputs land in `combination_search_drinks_outputs/` alongside the §1-§13 outputs.

> **Trading lens.** A pair-trade is a directional bet on a residual that mean-reverts. The niche signals below mostly *gate* or *enrich* such bets — e.g. "only enter when the vol-cluster sibling is calm," "trigger an A→B trade when A jumps," "use OBI[A] as a sign filter on the spread A−B."


In [ ]:
display(Markdown("""## §15 — Microstructure feature panel (beyond mid)

Up to here we've only used `mid_price`. The order book in `prices_round_5_day_*.csv` carries much richer per-tick features that often **leak information BEFORE the mid moves**:

| Feature | Definition | Why it matters |
|---|---|---|
| `bid1`, `ask1` | top of book | cleaner than mid for execution |
| `spread` | `ask1 − bid1` | volatility / illiquidity proxy |
| `bid_depth`, `ask_depth` | sum of L1+L2+L3 volumes | total committed liquidity per side |
| `obi` | `(bid_depth − ask_depth) / (bid_depth + ask_depth)` | imbalance ∈ [−1, 1]; classical short-horizon predictor |
| `microprice` | `(bid1·ask_vol1 + ask1·bid_vol1) / (bid_vol1 + ask_vol1)` | execution-weighted price |
| `mp_minus_mid` | `microprice − mid` | short-term drift signal |

Sections §20-§22 use these to look for cross-product information leakage that mid-correlation misses entirely."""))

# Reload prices with all columns (we previously kept only mid)
prices_full = pd.concat([
    pd.read_csv(f, sep=";").assign(
        day=int(re.search(r"day_(-?\d+)", f.stem).group(1))
            if re.search(r"day_(-?\d+)", f.stem) else 0
    )
    for f in sorted(DATA_DIR.glob("prices_round_*_day_*.csv"))
], ignore_index=True)
prices_full = prices_full[prices_full["product"].isin(PRODUCTS)].copy()
prices_full["t"] = prices_full["day"].astype(int) * 1_000_000 + prices_full["timestamp"].astype(int)

book_cols = ["bid_price_1", "ask_price_1", "bid_volume_1", "ask_volume_1",
             "bid_volume_2", "ask_volume_2", "bid_volume_3", "ask_volume_3",
             "bid_price_2", "ask_price_2", "bid_price_3", "ask_price_3"]
for c in book_cols:
    prices_full[c] = pd.to_numeric(prices_full[c], errors="coerce")

prices_full["bid_depth"] = prices_full[["bid_volume_1", "bid_volume_2", "bid_volume_3"]].sum(axis=1, min_count=1)
prices_full["ask_depth"] = prices_full[["ask_volume_1", "ask_volume_2", "ask_volume_3"]].sum(axis=1, min_count=1)
prices_full["spread"]    = prices_full["ask_price_1"] - prices_full["bid_price_1"]

top_v = prices_full["bid_volume_1"] + prices_full["ask_volume_1"]
prices_full["microprice"] = np.where(
    top_v > 0,
    (prices_full["bid_price_1"] * prices_full["ask_volume_1"]
     + prices_full["ask_price_1"] * prices_full["bid_volume_1"]) / top_v.replace(0, np.nan),
    prices_full["mid_price"],
)
denom = prices_full["bid_depth"] + prices_full["ask_depth"]
prices_full["obi"] = np.where(denom > 0,
    (prices_full["bid_depth"] - prices_full["ask_depth"]) / denom.replace(0, np.nan),
    0.0)
prices_full["mp_minus_mid"] = prices_full["microprice"] - prices_full["mid_price"]


def pivot_feature(df, feat):
    out = df.pivot_table(index="t", columns="product", values=feat, aggfunc="last")
    cols = [p for p in PRODUCTS if p in out.columns]
    return out[cols].sort_index().reindex(mid_aligned.index).ffill()


feat_panels = {
    "bid1":         pivot_feature(prices_full, "bid_price_1"),
    "ask1":         pivot_feature(prices_full, "ask_price_1"),
    "spread":       pivot_feature(prices_full, "spread"),
    "bid_depth":    pivot_feature(prices_full, "bid_depth"),
    "ask_depth":    pivot_feature(prices_full, "ask_depth"),
    "obi":          pivot_feature(prices_full, "obi"),
    "microprice":   pivot_feature(prices_full, "microprice"),
    "mp_minus_mid": pivot_feature(prices_full, "mp_minus_mid"),
}

print("Feature panels (rows = aligned ticks, cols = products):")
for name, panel in feat_panels.items():
    flat = panel.stack().dropna()
    print(f"  {name:14s} shape={panel.shape}  mean={flat.mean():+.4f}  std={flat.std():.4f}")

# Visual sanity
fig, axs = plt.subplots(2, 2, figsize=(14, 7))
feat_panels["spread"].plot(ax=axs[0, 0], lw=0.5, alpha=0.7)
axs[0, 0].set_title("Top-of-book spread"); axs[0, 0].legend(loc="upper right", fontsize=7)
feat_panels["obi"].plot(ax=axs[0, 1], lw=0.4, alpha=0.6)
axs[0, 1].set_title("Order-book imbalance"); axs[0, 1].legend(loc="upper right", fontsize=7)
feat_panels["mp_minus_mid"].plot(ax=axs[1, 0], lw=0.4, alpha=0.7)
axs[1, 0].set_title("microprice − mid (short-term drift)"); axs[1, 0].legend(loc="upper right", fontsize=7)
(feat_panels["bid_depth"] + feat_panels["ask_depth"]).plot(ax=axs[1, 1], lw=0.4, alpha=0.6)
axs[1, 1].set_title("Total book depth (bid + ask, L1+L2+L3)"); axs[1, 1].legend(loc="upper right", fontsize=7)
plt.tight_layout(); plt.show()

# Same-product OBI → return correlation as a sanity baseline
print("\nSame-product OBI(t) vs return(t+1) correlation (should be positive — buy-pressure leads price up):")
for p in columns:
    ob = feat_panels["obi"][p]
    rb = rets[p].shift(-1)
    m = (~ob.isna()) & (~rb.isna())
    if m.sum() > 100:
        print(f"  {p:30s}  corr = {ob[m].corr(rb[m]):+.4f}")


In [ ]:
display(Markdown("""## §16 — Non-linear dependence: distance correlation + mutual information

Pearson only sees **linear** dependence. Two series can be tightly coupled but Pearson-zero (e.g. `b ≈ a²`, or co-volatility without co-direction). We add two model-free measures:

- **Distance correlation (dCor)** — zero iff X and Y are independent. ∈ [0, 1]. Catches arbitrary monotone *and* non-monotone dependence.
- **Mutual information (MI)** — model-free dependence in nats; large MI with small Pearson is a smoking gun for non-linearity.

Both run on returns. Niche pairs = those with a large `dCor − |Pearson|` gap. Those are the ones where direct correlation is *lying to you*."""))

from sklearn.feature_selection import mutual_info_regression


def dcor(x, y, n_max=2000, seed=RANDOM_SEED):
    """Distance correlation between two 1-D arrays. Subsample to n_max for O(n²) RAM."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 50:
        return np.nan
    if len(x) > n_max:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(x), n_max, replace=False)
        x, y = x[idx], y[idx]
    a = np.abs(x[:, None] - x[None, :])
    b = np.abs(y[:, None] - y[None, :])
    A = a - a.mean(axis=0) - a.mean(axis=1)[:, None] + a.mean()
    B = b - b.mean(axis=0) - b.mean(axis=1)[:, None] + b.mean()
    dcov2  = (A * B).mean()
    dvar_x = (A * A).mean()
    dvar_y = (B * B).mean()
    if dvar_x <= 0 or dvar_y <= 0:
        return 0.0
    return float(np.sqrt(max(dcov2, 0.0) / np.sqrt(dvar_x * dvar_y)))


mi_subsample = min(len(rets), 5000)
rets_mi = rets.dropna()
if len(rets_mi) > mi_subsample:
    rets_mi = rets_mi.sample(n=mi_subsample, random_state=RANDOM_SEED).sort_index()

nl_rows = []
for a, b in combinations(columns, 2):
    pa = float(corr_ret.loc[a, b])
    dc = dcor(rets[a].values, rets[b].values)
    try:
        mi = float(mutual_info_regression(
            rets_mi[[a]].values, rets_mi[b].values,
            random_state=RANDOM_SEED, n_neighbors=5,
        )[0])
    except Exception:
        mi = np.nan
    nl_rows.append({
        "a": a, "b": b,
        "pearson_ret":          pa,
        "abs_pearson":          abs(pa),
        "dcor":                 dc,
        "mi":                   mi,
        "dcor_minus_pearson":   (dc if dc == dc else 0) - abs(pa),
    })

nonlin_table = pd.DataFrame(nl_rows).sort_values("dcor_minus_pearson", ascending=False).reset_index(drop=True)
nonlin_table.to_csv(OUT_DIR / "nonlinear_dependence.csv", index=False)

print("=== Non-linear dependence (sorted by dCor − |Pearson|) ===")
print("    high gap = pair has structure that linear correlation misses")
display(nonlin_table.round(4))

# Plot dCor vs |Pearson|; above the diagonal = non-linear edge
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
axs[0].scatter(nonlin_table["abs_pearson"], nonlin_table["dcor"], s=42, alpha=0.85)
mx = float(max(nonlin_table["abs_pearson"].max(), nonlin_table["dcor"].max())) * 1.05
axs[0].plot([0, mx], [0, mx], "k--", lw=0.7, label="dCor = |Pearson|")
for _, r in nonlin_table.iterrows():
    axs[0].annotate(f"{r['a'][-10:]}|{r['b'][-10:]}", (r["abs_pearson"], r["dcor"]),
                    fontsize=6.5, alpha=0.7)
axs[0].set_xlabel("|Pearson(ret)|"); axs[0].set_ylabel("dCor")
axs[0].set_title("Above the diagonal = non-linear edge"); axs[0].legend()

axs[1].bar(range(len(nonlin_table)), nonlin_table["mi"])
axs[1].set_xticks(range(len(nonlin_table)))
axs[1].set_xticklabels([f"{r['a'][-10:]}|{r['b'][-10:]}" for _, r in nonlin_table.iterrows()],
                        rotation=60, ha="right", fontsize=7)
axs[1].set_title("Mutual information (returns, KSG, k=5)"); axs[1].set_ylabel("MI (nats)")
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §17 — Tail / extreme co-movement (asymmetric, lagged)

Average correlation hides what happens in the tails. Define a **tail event** for product `p` at tick `t` as `|return_p(t)| > q_p` where `q_p` is the per-product 95th-percentile of `|returns|`. Then per ordered pair (a → b):

- `P(tail_b | tail_a)` — conditional probability of co-tail at the same tick
- `lift_any = P(tail_b | tail_a) / P(tail_b)` — lift vs independence
- Split into `up_b|up_a`, `dn_b|dn_a`, and the *cross-direction* `dn_b|up_a` to reveal **asymmetric** tail behaviour
- Lagged: `P(tail_b at t+1 | tail_a at t)` — does an extreme in A *precede* an extreme in B?

Tail dependence is invisible to Pearson, which averages the body of the distribution. A pair can have Pearson(ret)=0 but a 5× lift in joint extremes — a real, niche, tradeable shock-channel."""))

TAIL_PCT = 95


def tail_thr(s, pct):
    return float(np.nanpercentile(np.abs(s), pct))


tail_rows = []
for a in columns:
    ra = rets[a]
    th_a = tail_thr(ra, TAIL_PCT)
    up_a  = (ra >  th_a).fillna(False)
    dn_a  = (ra < -th_a).fillna(False)
    any_a = up_a | dn_a
    for b in columns:
        if a == b:
            continue
        rb = rets[b]
        th_b = tail_thr(rb, TAIL_PCT)
        up_b  = (rb >  th_b).fillna(False)
        dn_b  = (rb < -th_b).fillna(False)
        any_b = up_b | dn_b

        p_b = float(any_b.mean())
        cond_any   = float(any_b[any_a].mean()) if any_a.sum() else np.nan
        cond_up_up = float(up_b[up_a].mean())   if up_a.sum()  else np.nan
        cond_dn_dn = float(dn_b[dn_a].mean())   if dn_a.sum()  else np.nan
        cond_up_dn = float(dn_b[up_a].mean())   if up_a.sum()  else np.nan  # asymmetry: a-up → b-down

        any_a_lag1 = any_a.shift(1).fillna(False).astype(bool)
        cond_lag = float(any_b[any_a_lag1].mean()) if any_a_lag1.sum() else np.nan

        tail_rows.append({
            "a": a, "b": b,
            "p(tail_a)":         float(any_a.mean()),
            "p(tail_b)":         p_b,
            "p(tail_b|tail_a)":  cond_any,
            "lift_any":          (cond_any / p_b) if (cond_any == cond_any and p_b > 0) else np.nan,
            "p(up_b|up_a)":      cond_up_up,
            "p(dn_b|dn_a)":      cond_dn_dn,
            "p(dn_b|up_a)":      cond_up_dn,
            "lift_lagged":       (cond_lag / p_b) if (cond_lag == cond_lag and p_b > 0) else np.nan,
        })

tail_table = pd.DataFrame(tail_rows)
tail_table["asym_up_dn"] = tail_table["p(up_b|up_a)"] - tail_table["p(dn_b|dn_a)"]

print(f"=== Top contemporaneous tail co-movement (TAIL_PCT={TAIL_PCT}, lift > 1 = non-independent) ===")
display(tail_table.dropna(subset=["lift_any"]).sort_values("lift_any", ascending=False).head(20).round(3))

print("\n=== Asymmetric tail behaviour: |P(up_b|up_a) − P(dn_b|dn_a)| (top non-symmetric) ===")
display(tail_table.reindex(tail_table["asym_up_dn"].abs().sort_values(ascending=False).index).head(10).round(3))

print("\n=== Lagged tail predictability: tail in a at t → tail in b at t+1 ===")
display(tail_table.dropna(subset=["lift_lagged"]).sort_values("lift_lagged", ascending=False).head(10).round(3))

tail_table.to_csv(OUT_DIR / "tail_dependence.csv", index=False)


In [ ]:
display(Markdown("""## §18 — Volatility-of-volatility cross-product correlation

Two products can have **low return-correlation** but their **volatility** moves together — they get loud at the same time. Mid-correlation misses this entirely.

Compute correlation of:
- `|returns|` (absolute returns — classical vol proxy)
- `returns²` (variance contribution — emphasises tails)
- Rolling realized vol (window=`WINDOW_FAST`)

Niche pairs: low Pearson(ret) but high corr(|ret|) → **vol-cluster siblings**. Useful as a regime filter on §11 baskets — only enter pair-trades when the vol-sibling is calm, since a vol spike usually breaks the spread temporarily."""))

abs_rets = rets.abs()
sq_rets  = rets ** 2
roll_vol = rets.rolling(WINDOW_FAST, min_periods=50).std()

corr_abs = abs_rets.corr()
corr_sq  = sq_rets.corr()
corr_rv  = roll_vol.corr()

vol_rows = []
for a, b in combinations(columns, 2):
    pe = float(corr_ret.loc[a, b])
    ab = float(corr_abs.loc[a, b])
    sq = float(corr_sq.loc[a, b])
    rv = float(corr_rv.loc[a, b])
    vol_rows.append({
        "a": a, "b": b,
        "pearson_ret":   pe,
        "corr_abs_ret":  ab,
        "corr_sq_ret":   sq,
        "corr_roll_vol": rv,
        "vol_minus_ret": ab - abs(pe),
    })
vol_table = pd.DataFrame(vol_rows).sort_values("vol_minus_ret", ascending=False).reset_index(drop=True)
vol_table.to_csv(OUT_DIR / "volatility_correlation.csv", index=False)

print("=== Vol-correlation gap: corr(|ret|) − |Pearson(ret)| (high = vol cluster sibling) ===")
display(vol_table.round(4))

fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, C, t in [
    (axs[0], corr_abs, "corr |returns|"),
    (axs[1], corr_sq,  "corr returns²"),
    (axs[2], corr_rv,  f"corr rolling vol (w={WINDOW_FAST})"),
]:
    sns.heatmap(C, annot=True, fmt=".2f", center=0, vmin=-1, vmax=1, cmap="coolwarm",
                ax=ax, cbar=False, square=True)
    ax.set_title(t)
    ax.tick_params(axis="x", labelsize=7, rotation=45)
    ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()

# Side-by-side bar of pearson(ret) vs corr(|ret|) per pair
fig2, ax2 = plt.subplots(figsize=(11, 4))
labels = [f"{r['a'][-12:]}|{r['b'][-12:]}" for _, r in vol_table.iterrows()]
x = np.arange(len(vol_table))
ax2.bar(x - 0.2, vol_table["pearson_ret"].abs(),  width=0.4, label="|Pearson(ret)|")
ax2.bar(x + 0.2, vol_table["corr_abs_ret"],       width=0.4, label="corr(|ret|)")
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
ax2.set_title("Per-pair: linear return-corr vs absolute-return-corr (vol corr)")
ax2.legend(); ax2.set_ylim(0, 1)
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §19 — Sign-asymmetric lead-lag + Granger causality

The §5 lead-lag is symmetric: it doesn't distinguish whether `a` leads `b` only on **up-moves** or only on **down-moves**. Real edges are often one-sided — e.g. when a variant gets bought aggressively, the others follow within a tick, but on sells they don't.

We split returns:
- `ra_up = max(ra, 0)` — masks all down-moves to zero
- `ra_dn = min(ra, 0)` — masks all up-moves to zero

and compute `corr(ra_*, rb.shift(-L))` separately. The gap `up_corr − dn_corr` is the **directional asymmetry** — if it's strongly non-zero, you have a sign-conditional lead.

We then add **Granger causality F-tests** at lags 1, 5, 10. Granger has a known F-distribution under the null, so the p-values are interpretable for multiple-testing — unlike raw cross-correlation peaks."""))

from statsmodels.tsa.stattools import grangercausalitytests

LAG_AT = 1
asym_rows = []
for a in columns:
    ra = rets[a]
    ra_up = ra.where(ra > 0, 0.0)
    ra_dn = ra.where(ra < 0, 0.0)
    for b in columns:
        if a == b:
            continue
        rb_lead = rets[b].shift(-LAG_AT)
        m = (~ra.isna()) & (~rb_lead.isna())
        if m.sum() < 100:
            continue
        c_full = float(ra[m].corr(rb_lead[m]))
        c_up   = float(ra_up[m].corr(rb_lead[m])) if ra_up[m].std() > 0 else np.nan
        c_dn   = float(ra_dn[m].corr(rb_lead[m])) if ra_dn[m].std() > 0 else np.nan
        asym_rows.append({
            "a": a, "b": b, "lag": LAG_AT,
            "lead_corr_full":    c_full,
            "lead_corr_up_only": c_up,
            "lead_corr_dn_only": c_dn,
            "asym_up_minus_dn":  ((c_up if c_up == c_up else 0) - (c_dn if c_dn == c_dn else 0)),
        })
asym_table = pd.DataFrame(asym_rows)
asym_table = asym_table.reindex(asym_table["asym_up_minus_dn"].abs().sort_values(ascending=False).index).reset_index(drop=True)
asym_table.to_csv(OUT_DIR / "asymmetric_leadlag.csv", index=False)
print(f"=== Asymmetric lead-lag at L={LAG_AT} (sorted by |up − dn|) ===")
display(asym_table.round(4))

# Granger
GC_LAGS = [1, 5, 10]
gc_n = min(len(rets), 8000)
rets_gc = rets.dropna()
if len(rets_gc) > gc_n:
    rets_gc = rets_gc.sample(n=gc_n, random_state=RANDOM_SEED).sort_index()

gc_rows = []
for a in columns:
    for b in columns:
        if a == b:
            continue
        try:
            res = grangercausalitytests(rets_gc[[b, a]].values, maxlag=max(GC_LAGS), verbose=False)
            for L in GC_LAGS:
                p = float(res[L][0]["ssr_ftest"][1])
                F = float(res[L][0]["ssr_ftest"][0])
                gc_rows.append({"cause": a, "effect": b, "lag": L, "ssr_F": F, "ssr_F_p": p})
        except Exception as e:
            for L in GC_LAGS:
                gc_rows.append({"cause": a, "effect": b, "lag": L, "ssr_F": np.nan, "ssr_F_p": np.nan})

gc_table = pd.DataFrame(gc_rows)
gc_table.to_csv(OUT_DIR / "granger_causality.csv", index=False)
print("\n=== Granger causality (top 30 by lowest p) ===")
display(gc_table.sort_values("ssr_F_p").head(30).round(5))


In [ ]:
display(Markdown("""## §20 — Order-book imbalance → cross-product price predictability

OBI is a known short-horizon price predictor for the **same** product. Within a tightly correlated group, OBI in product A may **leak** information about product B *before* B's mid moves — quotes update in one variant and the rest follow.

For each ordered pair (a → b) compute:

`corr(obi[a]_t , Σ_{k=1..L} return[b]_{t+k})` for L ∈ {1, 5, 10, 20}

Sign matters: `+OBI` (more bid depth) should predict positive `b`-returns if there is real cross-product information. We also do `spread[a] → |return[b]|` as a vol-prediction control: a widening spread in A often anticipates volatility in correlated B."""))

obi_panel    = feat_panels["obi"]
spread_panel = feat_panels["spread"]

LEADS = [1, 5, 10, 20]
obi_rows = []
for a in columns:
    obi_a = obi_panel[a]
    sp_a  = spread_panel[a]
    for b in columns:
        rb = rets[b]
        row = {"a_obi": a, "b_ret": b}
        for L in LEADS:
            rb_fwd = rb.shift(-1).rolling(L, min_periods=max(1, L // 2)).sum().shift(-(L - 1))
            # Above is forward sum of returns over [t+1 .. t+L]; align to obi_a indexed at t
            m = (~obi_a.isna()) & (~rb_fwd.isna())
            row[f"corr_lead_{L}"] = float(obi_a[m].corr(rb_fwd[m])) if m.sum() > 100 else np.nan
        # spread → forward |return| (vol prediction)
        abs_b_fwd = rb.abs().shift(-1).rolling(5, min_periods=2).sum().shift(-4)
        m2 = (~sp_a.isna()) & (~abs_b_fwd.isna())
        row["spread_a_to_absret_b_lead5"] = float(sp_a[m2].corr(abs_b_fwd[m2])) if m2.sum() > 100 else np.nan
        obi_rows.append(row)

obi_pred = pd.DataFrame(obi_rows)
lead_cols = [f"corr_lead_{L}" for L in LEADS]
obi_pred["max_abs_lead"] = obi_pred[lead_cols].abs().max(axis=1)
obi_pred = obi_pred.sort_values("max_abs_lead", ascending=False).reset_index(drop=True)
obi_pred.to_csv(OUT_DIR / "obi_cross_predict.csv", index=False)

print("=== Same-product OBI → forward returns (baseline self-predictability) ===")
display(obi_pred.query("a_obi == b_ret").round(4))

print("\n=== Cross-product OBI → forward returns (the niche signal) ===")
display(obi_pred.query("a_obi != b_ret").round(4))

# Heatmap of one chosen lead
L_show = 5
heat = obi_pred.pivot(index="a_obi", columns="b_ret", values=f"corr_lead_{L_show}").reindex(index=columns, columns=columns)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(heat, annot=True, fmt=".3f", center=0, cmap="coolwarm", ax=ax,
            cbar_kws={"label": f"corr(obi[a]_t, Σ rets[b]_(t+1..t+{L_show}))"})
ax.set_title(f"{GROUP_LABEL}: cross-product OBI → forward return at L={L_show}")
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §21 — Co-jump synchronization

Define a **jump** for product `p` at tick `t` as `|return_p(t)| > k · σ_p(t)` where `σ_p(t)` is the rolling std (k=4, window=500 ticks). Then per pair (a, b):

- `n_a`, `n_b`: jump counts
- `n_ab`: simultaneous jumps at the same tick
- Expected under independence: `E[n_ab] = n_a · n_b / T`
- `lift = n_ab / E[n_ab]`,    `z = (n_ab − E) / √E`
- Lagged version: `a` jumps at `t`, `b` jumps at `t+1`

A `lift >> 1` means the pair shares a shock channel — same news, same liquidity event, etc. A high *lagged* lift is the more interesting signal: it says A's jump is a **trigger** for B's next-tick jump."""))

K_SIGMA  = 4.0
JUMP_WIN = 500
roll_sigma = rets.rolling(JUMP_WIN, min_periods=100).std()
jumps = (rets.abs() > K_SIGMA * roll_sigma).fillna(False).astype(int)

print(f"Per-product jump counts (k={K_SIGMA}σ, window={JUMP_WIN}):")
print(jumps.sum().to_string())
print(f"\nTotal usable ticks: {len(jumps):,}")

T_eff = (~roll_sigma.isna()).all(axis=1).sum()
cj_rows = []
for a, b in combinations(columns, 2):
    ja = jumps[a].astype(bool)
    jb = jumps[b].astype(bool)
    na = int(ja.sum()); nb = int(jb.sum())
    nab = int((ja & jb).sum())
    exp_ab = (na * nb / len(jumps)) if len(jumps) > 0 else 0.0
    lift   = (nab / exp_ab) if exp_ab > 0 else np.nan
    z      = (nab - exp_ab) / np.sqrt(exp_ab) if exp_ab > 0 else np.nan
    nab_lag = int((ja.shift(1).fillna(False).astype(bool) & jb).sum())
    lift_lag = (nab_lag / exp_ab) if exp_ab > 0 else np.nan
    nba_lag = int((jb.shift(1).fillna(False).astype(bool) & ja).sum())
    lift_lag_rev = (nba_lag / exp_ab) if exp_ab > 0 else np.nan
    cj_rows.append({
        "a": a, "b": b,
        "n_a": na, "n_b": nb,
        "n_ab_t0": nab, "exp_ab": exp_ab,
        "cojump_lift_t0": lift, "cojump_z_t0": z,
        "n_ab_lag1": nab_lag,    "cojump_lift_lag1": lift_lag,
        "n_ba_lag1": nba_lag,    "cojump_lift_rev_lag1": lift_lag_rev,
    })
cj_table = pd.DataFrame(cj_rows).sort_values("cojump_lift_t0", ascending=False).reset_index(drop=True)
cj_table.to_csv(OUT_DIR / "cojump_synchronization.csv", index=False)

print("\n=== Co-jump synchronization (lift > 1 = synced shocks) ===")
display(cj_table.round(3))

# Visualize jump times
fig, ax = plt.subplots(figsize=(13, 3.6))
for i, p in enumerate(columns):
    j_idx = np.where(jumps[p].values)[0]
    ax.scatter(j_idx, [i] * len(j_idx), s=10, alpha=0.6, label=f"{p} ({len(j_idx)})")
ax.set_yticks(range(len(columns))); ax.set_yticklabels([c[:24] for c in columns], fontsize=8)
ax.set_xlabel("aligned tick index")
ax.set_title(f"{GROUP_LABEL}: jump events (k={K_SIGMA}σ, win={JUMP_WIN})")
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §22 — Signed trade-flow cross-effects

Trades are committed demand, not just quotes. We sign each trade Lee-Ready style:

- `+qty` if `price ≥ ask₁` (aggressive buyer hit the ask)
- `−qty` if `price ≤ bid₁` (aggressive seller hit the bid)
- otherwise sign by `price` vs. mid (tick test)

Aggregate signed flow into the same tick grid (per product), then test cross-product predictability:

`corr(trade_flow[a]_t , return[b]_{t+L})` for L ∈ {0, 1, 5, 10}

This is the **executed-flow** version of the cross-OBI test in §20 — quote OBI is *intent*, signed flow is *action*. They can disagree: lots of resting bids that don't get hit (high OBI, low flow) vs. quiet quotes that get crossed every tick (low OBI, high flow)."""))

def load_trades(data_dir, products):
    files = sorted(data_dir.glob("trades_round_*_day_*.csv"))
    if not files:
        return pd.DataFrame(columns=["t", "symbol", "price", "quantity", "day", "timestamp"])
    frames = []
    for f in files:
        df = pd.read_csv(f, sep=";")
        m = re.search(r"day_(-?\d+)", f.stem)
        df["day"] = int(m.group(1)) if m else 0
        frames.append(df[df["symbol"].isin(products)])
    t = pd.concat(frames, ignore_index=True)
    t["t"] = t["day"].astype(int) * 1_000_000 + t["timestamp"].astype(int)
    return t


trades = load_trades(DATA_DIR, PRODUCTS)
print(f"Loaded {len(trades):,} trades across {trades['symbol'].nunique()} products on days {sorted(trades['day'].unique())}")

bid1_panel = feat_panels["bid1"]
ask1_panel = feat_panels["ask1"]
mid_panel  = mid_aligned

# Vectorised sign assignment via merge with the (t, symbol) book snapshot
book_long = pd.concat([
    bid1_panel.stack().rename("bid1"),
    ask1_panel.stack().rename("ask1"),
    mid_panel.stack().rename("mid"),
], axis=1).reset_index()
book_long.columns = ["t", "symbol", "bid1", "ask1", "mid"]

trades = trades.merge(book_long, on=["t", "symbol"], how="left")
sign = np.where(np.isfinite(trades["ask1"]) & (trades["price"] >= trades["ask1"]),  1,
        np.where(np.isfinite(trades["bid1"]) & (trades["price"] <= trades["bid1"]), -1,
         np.where(np.isfinite(trades["mid"])  & (trades["price"] >  trades["mid"]),  1,
          np.where(np.isfinite(trades["mid"]) & (trades["price"] <  trades["mid"]), -1, 0))))
trades["sign"] = sign
trades["signed_qty"] = trades["sign"] * trades["quantity"]

print("\nSign breakdown:")
print(trades.groupby("symbol")["sign"].value_counts().unstack(fill_value=0))

flow = trades.pivot_table(index="t", columns="symbol", values="signed_qty",
                          aggfunc="sum").reindex(mid_panel.index).fillna(0.0)
flow = flow[[c for c in PRODUCTS if c in flow.columns]]

print("\nPer-product signed-flow stats:")
display(pd.DataFrame({
    "n_trades":  trades.groupby("symbol").size().reindex(flow.columns, fill_value=0),
    "net_flow":  flow.sum(),
    "mean_abs":  flow.abs().mean(),
    "max_abs":   flow.abs().max(),
}).round(2))

LEADS_F = [0, 1, 5, 10]
flow_rows = []
for a in flow.columns:
    fa = flow[a]
    if fa.abs().sum() == 0:
        continue
    for b in columns:
        rb = rets[b]
        row = {"a_flow": a, "b_ret": b}
        for L in LEADS_F:
            if L == 0:
                rb_w = rb
            else:
                rb_w = rb.shift(-L).rolling(L, min_periods=max(1, L // 2)).sum()
            m = (~rb_w.isna()) & (fa != 0)
            row[f"corr_lead_{L}"] = float(fa[m].corr(rb_w[m])) if m.sum() > 50 else np.nan
        flow_rows.append(row)
flow_table = pd.DataFrame(flow_rows)
if len(flow_table):
    lead_cols_f = [f"corr_lead_{L}" for L in LEADS_F]
    flow_table["max_abs_corr"] = flow_table[lead_cols_f].abs().max(axis=1)
    flow_table = flow_table.sort_values("max_abs_corr", ascending=False).reset_index(drop=True)
    flow_table.to_csv(OUT_DIR / "trade_flow_cross.csv", index=False)
    print("\n=== Signed flow(a) → return(b) at L ∈ {0,1,5,10} ===")
    display(flow_table.round(4))
    print("\n=== Cross only (a ≠ b) — true information leakage ===")
    display(flow_table.query("a_flow != b_ret").round(4))
else:
    print("(no usable trade flow)")


In [ ]:
display(Markdown("""## §23 — Cross-spectral coherence (frequency-domain)

Coherence `C(f) = |P_xy(f)|² / (P_xx(f)·P_yy(f))` is the frequency-domain analogue of squared correlation. Pairs with the same time-domain Pearson can have very different coherence shapes:

- High coherence at **high frequency** = co-moves on every tick (microstructure / shared liquidity)
- High coherence at **low frequency** = co-trends over many ticks even when individual ticks decorrelate
- A frequency band where coherence peaks isolates the *time scale* the relationship lives at — directly tells you the holding horizon

We compute Welch's coherence per pair on returns and summarise by mean coherence in low / mid / high bands plus the peak."""))

from scipy.signal import coherence

NPERSEG = min(1024, max(64, len(rets) // 8))
fs = 1.0  # 1 sample per tick

coh_curves = {}
coh_rows = []
for a, b in combinations(columns, 2):
    f, Cxy = coherence(rets[a].fillna(0).values, rets[b].fillna(0).values, fs=fs, nperseg=NPERSEG)
    coh_curves[f"{a}|{b}"] = (f, Cxy)
    nfreq = len(f)
    if nfreq < 8:
        continue
    lo  = float(Cxy[1:nfreq // 4].mean())
    md_ = float(Cxy[nfreq // 4:nfreq // 2].mean())
    hi  = float(Cxy[nfreq // 2:].mean())
    peak_idx = 1 + int(np.argmax(Cxy[1:]))
    coh_rows.append({
        "a": a, "b": b,
        "coh_low": lo, "coh_mid": md_, "coh_hi": hi,
        "coh_peak": float(Cxy[peak_idx]),
        "coh_peak_freq": float(f[peak_idx]),
        "coh_peak_period_ticks": float(1.0 / f[peak_idx]) if f[peak_idx] > 0 else np.inf,
    })
coh_table = pd.DataFrame(coh_rows).sort_values("coh_peak", ascending=False).reset_index(drop=True)
coh_table.to_csv(OUT_DIR / "coherence_summary.csv", index=False)

print(f"=== Cross-spectral coherence (Welch, nperseg={NPERSEG}) ===")
display(coh_table.round(4))

# Plot every pair's coherence curve
fig = go.Figure()
for k, (fr, Cxy) in coh_curves.items():
    fig.add_trace(go.Scatter(x=fr, y=Cxy, name=k, mode="lines", line=dict(width=0.8)))
fig.update_layout(height=460,
                  title=f"{GROUP_LABEL}: cross-spectral coherence per pair",
                  xaxis_title="frequency (cycles per tick)",
                  yaxis_title="C(f)",
                  hovermode="x unified")
fig.show()


In [ ]:
display(Markdown("""## §24 — Niche dashboard + how to read

Compile the strongest **non-direct-correlation** signals from §16-§23 into one table. Each row is the top finding from one section. Use this as the entry point — drill into the underlying CSV for the full ranking."""))

niche = []

# §16 — strongest non-linear gap
if len(nonlin_table):
    r = nonlin_table.iloc[0]
    niche.append({"section": "§16 dCor − |Pearson|", "what": f"{r['a']} ↔ {r['b']}",
                  "score": r["dcor_minus_pearson"],
                  "metric": f"dCor={r['dcor']:.3f}, |Pearson|={r['abs_pearson']:.3f}"})

# §17 — strongest tail lift, both contemporaneous and lagged
if len(tail_table):
    r = tail_table.dropna(subset=["lift_any"]).sort_values("lift_any", ascending=False).iloc[0]
    niche.append({"section": "§17 tail co-move (contemp.)", "what": f"{r['a']} → {r['b']}",
                  "score": r["lift_any"],
                  "metric": f"P(tail_b|tail_a)={r['p(tail_b|tail_a)']:.3f}"})
    rl_pool = tail_table.dropna(subset=["lift_lagged"])
    if len(rl_pool):
        rl = rl_pool.sort_values("lift_lagged", ascending=False).iloc[0]
        niche.append({"section": "§17 tail lead (lag 1)", "what": f"{rl['a']} → {rl['b']}",
                      "score": rl["lift_lagged"],
                      "metric": "P(tail_b@t+1 | tail_a@t) lift"})

# §18 — biggest gap between |ret| and ret correlation
if len(vol_table):
    r = vol_table.iloc[0]
    niche.append({"section": "§18 vol-cluster sibling", "what": f"{r['a']} ↔ {r['b']}",
                  "score": r["vol_minus_ret"],
                  "metric": f"corr(|ret|)={r['corr_abs_ret']:.3f}, Pearson(ret)={r['pearson_ret']:.3f}"})

# §19 — strongest asymmetric leader, strongest Granger
if len(asym_table):
    r = asym_table.iloc[0]
    niche.append({"section": "§19 asymmetric lead-lag", "what": f"{r['a']} → {r['b']}",
                  "score": r["asym_up_minus_dn"],
                  "metric": f"up={r['lead_corr_up_only']:.3f}, dn={r['lead_corr_dn_only']:.3f}"})
gc_ok = gc_table.dropna(subset=["ssr_F_p"]) if "gc_table" in dir() else pd.DataFrame()
if len(gc_ok):
    r = gc_ok.sort_values("ssr_F_p").iloc[0]
    niche.append({"section": "§19 best Granger",
                  "what": f"{r['cause']} ⇒ {r['effect']} (lag {int(r['lag'])})",
                  "score": -np.log10(max(r["ssr_F_p"], 1e-300)),
                  "metric": f"p={r['ssr_F_p']:.2e}"})

# §20 — strongest cross-OBI predictor (excluding self)
if len(obi_pred):
    cross = obi_pred.query("a_obi != b_ret")
    lead_cols = [c for c in obi_pred.columns if c.startswith("corr_lead_")]
    cross = cross.assign(_best=cross[lead_cols].abs().max(axis=1)).sort_values("_best", ascending=False)
    if len(cross):
        r = cross.iloc[0]
        best_L = max(lead_cols, key=lambda c: abs(r[c]) if r[c] == r[c] else 0)
        niche.append({"section": "§20 OBI(a) → ret(b) cross",
                      "what": f"{r['a_obi']} obi → {r['b_ret']} ret",
                      "score": float(r[best_L]),
                      "metric": f"{best_L}={r[best_L]:.3f}"})

# §21 — strongest co-jump
if len(cj_table):
    r = cj_table.iloc[0]
    niche.append({"section": "§21 co-jump (contemp.)", "what": f"{r['a']} ↔ {r['b']}",
                  "score": r["cojump_lift_t0"] if r["cojump_lift_t0"] == r["cojump_lift_t0"] else 0,
                  "metric": f"n_ab={r['n_ab_t0']}, z={r['cojump_z_t0']:.1f}"})
    if cj_table["cojump_lift_lag1"].notna().any():
        rl = cj_table.dropna(subset=["cojump_lift_lag1"]).sort_values("cojump_lift_lag1", ascending=False).iloc[0]
        niche.append({"section": "§21 co-jump lead (lag 1)",
                      "what": f"{rl['a']} → {rl['b']}",
                      "score": rl["cojump_lift_lag1"],
                      "metric": f"n_ab_lag1={rl['n_ab_lag1']}"})

# §22 — strongest cross trade-flow predictor
if "flow_table" in dir() and len(flow_table):
    cross_flow = flow_table.query("a_flow != b_ret").copy()
    if len(cross_flow):
        lead_cols_f = [c for c in flow_table.columns if c.startswith("corr_lead_")]
        cross_flow["_best"] = cross_flow[lead_cols_f].abs().max(axis=1)
        cross_flow = cross_flow.sort_values("_best", ascending=False)
        r = cross_flow.iloc[0]
        best_L = max(lead_cols_f, key=lambda c: abs(r[c]) if r[c] == r[c] else 0)
        niche.append({"section": "§22 trade-flow(a) → ret(b)",
                      "what": f"{r['a_flow']} flow → {r['b_ret']} ret",
                      "score": float(r[best_L]),
                      "metric": f"{best_L}={r[best_L]:.3f}"})

# §23 — strongest peak coherence and biggest low/high gap
if len(coh_table):
    r = coh_table.iloc[0]
    niche.append({"section": "§23 peak coherence", "what": f"{r['a']} ↔ {r['b']}",
                  "score": r["coh_peak"],
                  "metric": f"peak f={r['coh_peak_freq']:.4f}, lo={r['coh_low']:.3f} hi={r['coh_hi']:.3f}"})
    coh_with_gap = coh_table.assign(lo_minus_hi=lambda d: d["coh_low"] - d["coh_hi"])
    rg = coh_with_gap.reindex(coh_with_gap["lo_minus_hi"].abs().sort_values(ascending=False).index).iloc[0]
    niche.append({"section": "§23 freq-band imbalance",
                  "what": f"{rg['a']} ↔ {rg['b']}",
                  "score": rg["lo_minus_hi"],
                  "metric": f"low={rg['coh_low']:.3f}, high={rg['coh_hi']:.3f}"})

niche_df = pd.DataFrame(niche)
niche_df.to_csv(OUT_DIR / "niche_dashboard.csv", index=False)
display(Markdown("### Niche dashboard — strongest non-direct-correlation findings"))
display(niche_df.round(4))

display(Markdown("""
### How to read these niche signals

| Signal | What it means in trading | Where to look next |
|---|---|---|
| **High dCor, low Pearson (§16)** | Pair has structure but not linear — vol-coupling, regime-switch, or non-monotone | Pair scatter plots in §10; check feature transformations |
| **Tail lift > 2 (§17)** | Extreme moves in A really do drag B. Asymmetric tail (`up_b\\|up_a` ≠ `dn_b\\|dn_a`) implies a tradeable directional leak | Build a "post-tail" event-study: when A jumps, what's the avg L+1..L+5 return of B? |
| **Vol-cluster sibling (§18)** | Pair shares volatility regime but not direction — useful for **vol breakout** triggers | Use as a regime filter on the §11 integer baskets |
| **Asymmetric lead-lag (§19)** | A leads B *only* one direction — one-sided liquidity / inventory effect | Trade only the asymmetric direction |
| **Granger causality (§19)** | A's history improves prediction of B beyond B's own history. Confirmatory test for §5 lead-lag | Use before staking on a raw cross-corr peak |
| **Cross-OBI lead (§20)** | A's quote imbalance leaks into B's price. Read A's book to predict B's next-tick move | At every tick where `obi_a` is extreme, take a directional bet on B |
| **Co-jump lift (§21)** | When one jumps, the other jumps too. Lagged → A's jump triggers B's jump at t+1 | Use A's jump as a *trigger* for an immediate B trade |
| **Trade-flow cross (§22)** | A's executed flow predicts B's return — stronger than quote OBI because flow is committed demand | Most promising for directional alpha; PnL-attribute vs §11 baskets |
| **Coherence at high f (§23)** | Pair shares microstructure noise — bad for tick-pair-trading but fine for slow baskets | If `coh_low` >> `coh_hi`, the **slow** basket is the right unit |

A real edge needs all three:
1. Statistical significance after multiple-testing inflation (§16/§17/§21 each tested 10+ pairs)
2. A residual / event-study plot with many round-trips (§13 logic)
3. A plausible micro-explanation (you should be able to *say* why the relationship exists)
"""))


# Part III — Multi-product strategies (niche)

Sections §1-§14 are direct-correlation pair/basket machinery. §15-§24 explored non-direct-correlation pair signals (microstructure, tail, vol, frequency, OBI, trade flow). This third part attacks the question that's only meaningful with **multiple products at once**:

> Treat the 5-product system as a *system*, not 10 separate pairs.

| §  | What we test | Why a *multi-product* approach beats pairs |
|----|---|---|
| §25 | k-of-N tail confluence + multi-product jump event-study | Pair-tail counts 2-of-5; the rare 3-of-5 / 4-of-5 ticks are where the real shocks live |
| §26 | Cross-sectional dispersion mean-reversion | Each product against the *equal-weighted* basket of the other 4 — naturally integer, no fitted hedge |
| §27 | Johansen multi-product cointegration | Finds *how many* independent stationary baskets (rank ≥ 2 = multiple parallel spreads) |
| §28 | Sparse Lasso-VAR | Returns the directed lead-lag *network* after Lasso-pruning mediated edges |
| §29 | Partial correlations + minimum spanning tree | Strips out 3-way mediation; identifies hub vs. leaf products |
| §30 | Diebold-Yilmaz volatility / return spillover | Quantifies leader/follower roles via FEVD; total spillover index = regime indicator |
| §31 | PCA cross-prediction | Lagged corr between PCs — does PC2 lead PC1? Niche residual-leads-trend signal |
| §32 | Regime-conditional baskets | Re-fits §6/§11 spreads under vol/dispersion/OBI regimes — only trade in the regime where the spread is stationary |
| §33 | Trade-flow cross-excitation | Lagged trade-burst correlation — bot mirroring across variants |
| §34 | Multi-product niche dashboard | Roll-up of §25-§33 winners with concrete trading lenses |

Most of these depend on the §15 microstructure panel and the §22 trade flow panel, so run §1-§24 first. Heavy compute is in §28 (Lasso CV per target) and §30 (VAR fit) — both kept under ~1 minute by capping subsample to 10-20 k ticks. Outputs land in `combination_search_drinks_outputs/` alongside everything else.


In [ ]:
display(Markdown("""## §25 — k-of-N tail confluence + multi-product jump event-study

Pair-tail dependence (§17) catches 2-product co-extremes. The richer signal is **how many** of the 5 products are extreme together at the same tick — the **confluence count** `K(t) = Σᵢ 1{|ret_i(t)| > q_i}`. Under independence, `K(t) ~ Binomial(N, p)` where `p ≈ (100 − TAIL_PCT)/100`.

We compare the empirical distribution of K to the binomial null:
- `n_obs(k)` = ticks with exactly k tail products
- `n_exp(k)` = expected under independence
- `lift(k) = n_obs(k) / n_exp(k)` — high lift at large k = strong group-shock channel

Then a **multi-product jump event-study**: for every tick where ANY product jumps, compute the average return of every product in the window [t−5, t+5]. Reveals the typical group response — whether others lead, lag, or react idiosyncratically — and quantifies the **conditional impulse** of each trigger."""))

from scipy.stats import binom

# k-of-N tail confluence
TAIL_PCT_M = 95
thresholds = {p: float(np.nanpercentile(rets[p].abs().values, TAIL_PCT_M)) for p in columns}
tail_mat = pd.DataFrame({p: (rets[p].abs() > thresholds[p]) for p in columns}).fillna(False)
K = tail_mat.sum(axis=1).astype(int)

p_ind = (100.0 - TAIL_PCT_M) / 100.0
T_eff = len(K)

k_rows = []
for k in range(0, P + 1):
    n_obs = int((K == k).sum())
    p_exp = float(binom.pmf(k, P, p_ind))
    n_exp = T_eff * p_exp
    lift  = (n_obs / n_exp) if n_exp > 0 else np.nan
    z     = (n_obs - n_exp) / np.sqrt(n_exp * (1 - p_exp)) if n_exp > 0 and p_exp < 1 else np.nan
    k_rows.append({"k": k, "n_obs": n_obs, "n_exp": n_exp, "lift": lift, "z_score": z})
k_table = pd.DataFrame(k_rows)
k_table.to_csv(OUT_DIR / "k_of_N_tail.csv", index=False)
print(f"=== k-of-N tail confluence (TAIL_PCT={TAIL_PCT_M}, p_ind={p_ind:.3f}) ===")
print("    k = # products simultaneously in tail at the same tick")
display(k_table.round(3))

# Visualise
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].bar(k_table["k"] - 0.18, k_table["n_obs"], width=0.36, label="observed")
axs[0].bar(k_table["k"] + 0.18, k_table["n_exp"], width=0.36, label="binomial null")
axs[0].set_yscale("log"); axs[0].set_xlabel("k = # tail products"); axs[0].set_ylabel("count of ticks (log)")
axs[0].set_title("Observed vs. independent binomial"); axs[0].legend()
axs[1].bar(k_table["k"], k_table["lift"], color="steelblue")
axs[1].axhline(1, color="black", lw=0.7, linestyle="--")
axs[1].set_xlabel("k"); axs[1].set_ylabel("lift = obs / exp"); axs[1].set_title("Confluence lift")
plt.tight_layout(); plt.show()

# ------------------------------------------------------------------
# Multi-product jump event study — when ANY product jumps, what does the group do?
JUMP_K_SIG = 4.0
JUMP_WIN_M = 500
roll_sigma_m = rets.rolling(JUMP_WIN_M, min_periods=100).std()
jump_mat = (rets.abs() > JUMP_K_SIG * roll_sigma_m).fillna(False)

WINDOW = 5
event_rows = []
ret_arr_evt = rets.values
for ti, trigger in enumerate(columns):
    j_idx = np.where(jump_mat[trigger].values)[0]
    if len(j_idx) == 0:
        continue
    # keep only events with full pre/post window
    j_idx = j_idx[(j_idx >= WINDOW) & (j_idx < len(rets) - WINDOW - 1)]
    if len(j_idx) == 0:
        continue
    for ri, response in enumerate(columns):
        windows = np.array([ret_arr_evt[j - WINDOW: j + WINDOW + 1, ri] for j in j_idx])
        avg_resp = np.nanmean(windows, axis=0)
        cum_resp = np.cumsum(avg_resp)
        for offset, val, cum in zip(range(-WINDOW, WINDOW + 1), avg_resp, cum_resp):
            event_rows.append({
                "trigger":   trigger,
                "response":  response,
                "lag":       offset,
                "n_events":  int(len(j_idx)),
                "avg_ret":   float(val),
                "cum_ret":   float(cum),
            })

event_table = pd.DataFrame(event_rows)
event_table.to_csv(OUT_DIR / "jump_event_study.csv", index=False)

print(f"\n=== Jump event study (k={JUMP_K_SIG}σ trigger, ±{WINDOW} ticks) ===")
print("    cumulative response after t=0 (post-jump drift in each response product per trigger)")
post_pivot = event_table.query("lag >= 0").groupby(["trigger", "response"])["avg_ret"].sum().unstack().round(5)
display(post_pivot)

# Subplot per trigger
fig, axs = plt.subplots(1, P, figsize=(3.6 * P, 4.0), sharey=True)
for i, trigger in enumerate(columns):
    sub = event_table[event_table["trigger"] == trigger]
    ax = axs[i] if P > 1 else axs
    if not len(sub):
        ax.set_title(f"{trigger[-12:]}\n(no jumps)", fontsize=9)
        continue
    for response in columns:
        s = sub[sub["response"] == response].sort_values("lag")
        if not len(s): continue
        lw = 1.6 if response == trigger else 0.9
        ax.plot(s["lag"], s["cum_ret"], label=response[-12:], lw=lw)
    ax.axvline(0, ls="--", color="black", lw=0.7); ax.axhline(0, ls=":", color="grey", lw=0.5)
    n_ev = int(sub['n_events'].iloc[0])
    ax.set_title(f"trigger: {trigger[-14:]}\nn={n_ev}", fontsize=9)
    ax.set_xlabel("lag (ticks from jump)")
if P > 1:
    axs[0].set_ylabel("cumulative avg return")
    axs[0].legend(fontsize=6, loc="upper left")
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §26 — Cross-sectional dispersion mean-reversion

The classic multi-product strategy: at each tick, compute each product's z-score within the cross-section and bet on reversion.

1. Z-score each product's mid: `z_i(t) = (mid_i(t) − μ_i) / σ_i` (per-product time normalisation)
2. Cross-sectional mean and std at tick t: `cs_mean(t) = mean_i z_i(t)`,  `cs_std(t) = stdev_i z_i(t)`
3. **Cross-sectional position** of product i: `r_i(t) = (z_i(t) − cs_mean(t)) / cs_std(t)`

When `|r_i(t)|` is large, i is the group **outlier**. Hypothesis: outliers mean-revert toward the group. The trade is **(short i) vs. (long equal-weight basket of the other N−1)** — automatically integer-friendly because the basket weight is uniform.

We measure: autocorrelation of `r_i(t)`, ADF p-value, half-life. Low ADF p + short half-life = a tradeable group-relative-value edge."""))

z_mid_xs = (mid_aligned - mid_aligned.mean()) / mid_aligned.std(ddof=0).replace(0, np.nan)
cs_mean_xs = z_mid_xs.mean(axis=1)
cs_std_xs  = z_mid_xs.std(axis=1, ddof=0).replace(0, np.nan)
r_xs = z_mid_xs.sub(cs_mean_xs, axis=0).div(cs_std_xs, axis=0)

# The naive trade: product − equal-weight basket of others
basket_others = {p: mid_aligned.drop(columns=p).mean(axis=1) for p in columns}
xs_resid = pd.DataFrame({p: mid_aligned[p] - basket_others[p] for p in columns})

xs_rows = []
for p in columns:
    s_r = r_xs[p].dropna()
    s_e = xs_resid[p].dropna()
    xs_rows.append({
        "product":               p,
        "r_xs_std":              float(s_r.std()),
        "r_xs_acf_lag1":         float(s_r.autocorr(1)),
        "r_xs_adf_p":            adf_p(s_r.values),
        "r_xs_half_life":        half_life(s_r.values),
        "naive_resid_adf_p":     adf_p(s_e.values),
        "naive_resid_half_life": half_life(s_e.values),
    })
xs_table = pd.DataFrame(xs_rows).sort_values("r_xs_adf_p").reset_index(drop=True)
xs_table.to_csv(OUT_DIR / "cross_sectional_dispersion.csv", index=False)
print("=== Cross-sectional dispersion: each product as group outlier ===")
display(xs_table.round(4))

# Plot all 5 cross-sectional positions
fig = make_subplots(rows=P, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[
                        f"r_xs[{p}]   ADF p="
                        f"{(xs_table.set_index('product').loc[p, 'r_xs_adf_p'] if p in xs_table['product'].values else np.nan):.3g}, "
                        f"hl="
                        f"{(xs_table.set_index('product').loc[p, 'r_xs_half_life'] if p in xs_table['product'].values else np.nan)}"
                        for p in columns
                    ])
for i, p in enumerate(columns):
    fig.add_trace(go.Scatter(y=r_xs[p].values, line=dict(width=0.7), name=p), row=i + 1, col=1)
    fig.add_hline(y=0,  line_dash="dot",   line_color="black", row=i + 1, col=1)
    fig.add_hline(y=2,  line_dash="dot",   line_color="red",   row=i + 1, col=1)
    fig.add_hline(y=-2, line_dash="dot",   line_color="red",   row=i + 1, col=1)
fig.update_layout(height=160 * P, showlegend=False, hovermode="x unified",
                  title=f"{GROUP_LABEL}: cross-sectional position r_i(t) per product (±2 lines = entry triggers)")
fig.show()

# Daily distribution of |r_xs| — when is the group most dispersed?
fig2, ax2 = plt.subplots(figsize=(11, 3.5))
disp_series = cs_std_xs.dropna()
ax2.plot(disp_series.values, lw=0.5)
ax2.set_title("Cross-sectional dispersion cs_std(t) — high = group fragmenting")
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §27 — Johansen multi-product cointegration

Engle-Granger (used in §6) tests pairs. For a 5-product system the right tool is the **Johansen test**, which finds the **rank** of the cointegration system — i.e. *how many independent stationary linear combinations* exist:

- Rank r = 0: no cointegration (random-walk system)
- Rank r = 1: a single shared stationary basket
- Rank r ∈ {2, 3, 4}: multiple independent stationary baskets (rich relative-value structure)
- Rank r = 5: the level itself is stationary (rare)

For each rank r we report the **trace statistic** with its 95% critical value. The largest r that rejects H₀(r ≤ k) is the system's cointegration rank. Each cointegrating eigenvector is a multi-product spread that mean-reverts — so rank-r systems have r independent spreads to trade in parallel."""))

from statsmodels.tsa.vector_ar.vecm import coint_johansen

JOHN_SUBSAMPLE = 10000
X_jo = mid_aligned.values
if len(X_jo) > JOHN_SUBSAMPLE:
    idx = np.linspace(0, len(X_jo) - 1, JOHN_SUBSAMPLE).astype(int)
    X_jo = X_jo[idx]

try:
    jres = coint_johansen(X_jo, det_order=0, k_ar_diff=1)
    trace_table = pd.DataFrame({
        "r ≤":             list(range(P)),
        "trace_stat":      jres.lr1,
        "crit_90":         jres.cvt[:, 0],
        "crit_95":         jres.cvt[:, 1],
        "crit_99":         jres.cvt[:, 2],
        "max_eig_stat":    jres.lr2,
        "max_eig_crit_95": jres.cvm[:, 1],
    })
    print("=== Johansen trace test (reject H0:r≤k if trace_stat > crit_95) ===")
    display(trace_table.round(3))

    inferred_rank = int(np.sum(jres.lr1 > jres.cvt[:, 1]))
    print(f"\nInferred cointegration rank (trace test, 95% CV): r = {inferred_rank}")

    eig_df = pd.DataFrame(jres.evec, index=columns,
                          columns=[f"v{i+1}" for i in range(P)])
    print("\n=== Cointegrating eigenvectors (cols = baskets, rows = products) ===")
    display(eig_df.round(4))

    # For each eigenvector basket, compute the basket time series + ADF + half-life
    jrows = []
    for i in range(P):
        v = jres.evec[:, i]
        basket = mid_aligned.values @ v
        jrows.append({
            "eigvec":      f"v{i+1}",
            "weights":     {p: round(float(w), 4) for p, w in zip(columns, v)},
            "basket_std":  float(np.std(basket)),
            "adf_p":       adf_p(basket),
            "half_life":   half_life(basket),
        })
    jbasket_table = pd.DataFrame(jrows)
    jbasket_table.to_csv(OUT_DIR / "johansen_baskets.csv", index=False)
    print("\n=== Johansen baskets — ADF on each eigenvector spread ===")
    display(jbasket_table)

    # Plot the level series for the most stationary eigenvector basket
    rok = jbasket_table.dropna(subset=["adf_p"]).sort_values("adf_p")
    if len(rok):
        best = rok.iloc[0]
        v = jres.evec[:, int(best["eigvec"][1:]) - 1]
        basket_series = pd.Series(mid_aligned.values @ v, index=mid_aligned.index)
        roll_m = basket_series.rolling(WINDOW_FAST, min_periods=50).mean()
        roll_s = basket_series.rolling(WINDOW_FAST, min_periods=50).std()
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                            subplot_titles=[
                                f"Most stationary Johansen basket: {best['eigvec']} "
                                f"(ADF p={best['adf_p']:.3g}, half-life={best['half_life']})",
                                "z-score",
                            ])
        fig.add_trace(go.Scatter(y=(roll_m + 2*roll_s).values, line=dict(width=0), showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(y=(roll_m - 2*roll_s).values, line=dict(width=0), fill="tonexty",
                                 fillcolor="rgba(255,165,0,0.15)", showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(y=basket_series.values, line=dict(width=0.7, color="steelblue"), name="basket"), row=1, col=1)
        z_b = (basket_series - roll_m) / roll_s
        fig.add_trace(go.Scatter(y=z_b.values, line=dict(width=0.7, color="purple"), name="z"), row=2, col=1)
        fig.update_layout(height=520, showlegend=False, hovermode="x unified",
                          title="Best Johansen-cointegrated basket — residual + z-score")
        fig.show()
except Exception as e:
    print(f"Johansen failed: {e}")
    jbasket_table = pd.DataFrame()


In [ ]:
display(Markdown("""## §28 — Sparse vector autoregression (Lasso-VAR) — directed lead-lag network

A vector autoregression `r(t) = A_1 r(t−1) + A_2 r(t−2) + … + A_L r(t−L) + ε` describes the entire 5-product return system. With **Lasso** regularisation, most entries of `A_l` are forced to zero, leaving only the **directed edges** (i, j, lag) where one product's lag genuinely improves prediction of another's return — and crucially, edges that survive after **controlling for all other products simultaneously**.

This is much stronger than the §5 pair-wise lead-lag because it controls for confounds: a Pearson lead might be entirely mediated through a third variant. Lasso strips those mediated edges away and leaves the irreducible network."""))

from sklearn.linear_model import LassoCV


def make_var_X(rets_df, L):
    parts = []
    names = []
    for l in range(1, L + 1):
        parts.append(rets_df.shift(l).rename(columns=lambda c: f"{c}_lag{l}"))
        names.extend([f"{c}_lag{l}" for c in rets_df.columns])
    X_lag = pd.concat(parts, axis=1).dropna()
    return X_lag, names


VAR_LAGS = 5
X_lag, feat_names = make_var_X(rets, VAR_LAGS)
y_aligned = rets.loc[X_lag.index]

VAR_SUBSAMPLE = 10000
if len(X_lag) > VAR_SUBSAMPLE:
    rng = np.random.default_rng(RANDOM_SEED)
    idx = sorted(rng.choice(len(X_lag), VAR_SUBSAMPLE, replace=False))
    X_lag = X_lag.iloc[idx]
    y_aligned = y_aligned.iloc[idx]

print(f"Fitting LassoCV per target  (n={len(X_lag)}, p={X_lag.shape[1]}, lags=1..{VAR_LAGS}) ...")

network_rows = []
coef_mats = {l: pd.DataFrame(0.0, index=columns, columns=columns) for l in range(1, VAR_LAGS + 1)}
r2_per_target = {}

for target in columns:
    y = y_aligned[target].values
    m = ~np.isnan(y) & np.isfinite(X_lag.values).all(axis=1)
    if m.sum() < 100:
        continue
    model = LassoCV(cv=4, n_alphas=20, random_state=RANDOM_SEED, max_iter=3000, n_jobs=-1)
    model.fit(X_lag.values[m], y[m])
    coefs = pd.Series(model.coef_, index=feat_names)
    r2_per_target[target] = float(model.score(X_lag.values[m], y[m]))
    for fname, val in coefs.items():
        if abs(val) < 1e-9:
            continue
        cause, lag_str = fname.rsplit("_lag", 1)
        lag = int(lag_str)
        coef_mats[lag].at[cause, target] = val
        if cause != target:
            network_rows.append({
                "cause":   cause,
                "effect":  target,
                "lag":     lag,
                "coef":    float(val),
                "abs_coef": float(abs(val)),
            })

print("\nLasso-VAR R² per target (in-sample):")
for k, v in r2_per_target.items():
    print(f"  {k:30s}  R² = {v:+.4f}")

network_table = pd.DataFrame(network_rows).sort_values("abs_coef", ascending=False).reset_index(drop=True)
network_table.to_csv(OUT_DIR / "lasso_var_network.csv", index=False)
print("\n=== Lasso-VAR directed network (cross-edges only, sorted by |coef|) ===")
display(network_table.head(40).round(5))

# Heatmaps per lag
fig, axs = plt.subplots(1, VAR_LAGS, figsize=(3.5 * VAR_LAGS, 4))
for l in range(1, VAR_LAGS + 1):
    ax = axs[l - 1] if VAR_LAGS > 1 else axs
    sns.heatmap(coef_mats[l], annot=True, fmt=".3f", center=0, cmap="coolwarm",
                cbar=False, ax=ax, square=True)
    ax.set_title(f"VAR coef at lag {l}\nrows=cause, cols=effect", fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', labelsize=6)
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §29 — Partial correlations + minimum spanning tree

Pearson correlation between A and B can be **entirely mediated** through a third variable C — A and B only co-move because both move with C. **Partial correlation** removes this confounding: `partial_corr(A, B | rest) = corr` of the residuals after regressing both A and B on the other products.

This isolates **direct** dependencies. We then build the **minimum spanning tree** using `1 − |partial_corr|` as edge length: the MST is the skeleton of *direct* relationships in the group.

Graph-theoretic insight:
- **Hub** (high MST degree) = group's information carrier
- **Leaf** = follower; its idiosyncratic deviation is the most likely thing to mean-revert"""))

# Partial correlation matrix from inverse covariance (precision matrix)
ret_arr_full = rets.dropna().values
ret_cov = np.cov(ret_arr_full.T)
try:
    prec = np.linalg.inv(ret_cov)
    diag_p = np.sqrt(np.abs(np.diag(prec)))
    partial = -prec / np.outer(diag_p, diag_p)
    np.fill_diagonal(partial, 1.0)
    partial_df = pd.DataFrame(partial, index=columns, columns=columns)
except np.linalg.LinAlgError:
    print("(precision matrix singular — falling back to Pearson)")
    partial_df = corr_ret.copy()

print("=== Partial correlation matrix (corr after removing all other products) ===")
display(partial_df.round(4))

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(corr_ret, annot=True, fmt=".3f", center=0, vmin=-1, vmax=1, cmap="coolwarm",
            ax=axs[0], cbar=False, square=True)
axs[0].set_title("Pearson(returns) — pair-wise (includes mediated)")
axs[0].tick_params(axis="x", labelsize=7, rotation=45); axs[0].tick_params(axis="y", labelsize=7)
sns.heatmap(partial_df, annot=True, fmt=".3f", center=0, vmin=-1, vmax=1, cmap="coolwarm",
            ax=axs[1], cbar=False, square=True)
axs[1].set_title("Partial correlation — direct (others removed)")
axs[1].tick_params(axis="x", labelsize=7, rotation=45); axs[1].tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()

from scipy.sparse.csgraph import minimum_spanning_tree

dist = 1.0 - partial_df.abs().values
np.fill_diagonal(dist, 0.0)
dist = np.clip(dist, 0.0, None)
mst = minimum_spanning_tree(dist).toarray()
mst_edges = []
for i in range(len(columns)):
    for j in range(len(columns)):
        if mst[i, j] > 0:
            mst_edges.append({
                "a": columns[i], "b": columns[j],
                "partial_corr": float(partial_df.iat[i, j]),
                "edge_len":     float(mst[i, j]),
            })
mst_df = pd.DataFrame(mst_edges)
mst_df.to_csv(OUT_DIR / "partial_corr_mst.csv", index=False)
print("\n=== Minimum spanning tree edges (distance = 1 − |partial_corr|) ===")
display(mst_df.round(4))

# Node degree
degree = pd.Series(0, index=columns, dtype=int)
for _, e in mst_df.iterrows():
    degree[e["a"]] += 1
    degree[e["b"]] += 1
print("\n=== Node degree in MST (highest = group's information hub) ===")
print(degree.sort_values(ascending=False).to_string())

# Pair table: pearson vs partial — finds spurious pairs
pair_compare = []
for a, b in combinations(columns, 2):
    pe = float(corr_ret.loc[a, b])
    pa = float(partial_df.loc[a, b])
    pair_compare.append({
        "a": a, "b": b,
        "pearson": pe,
        "partial": pa,
        "drop":    abs(pe) - abs(pa),    # high = pearson is mediated
    })
pair_compare_df = pd.DataFrame(pair_compare).sort_values("drop", ascending=False).reset_index(drop=True)
print("\n=== Pearson minus partial: high = pair correlation is mostly mediated through others ===")
display(pair_compare_df.round(4))


In [ ]:
display(Markdown("""## §30 — Diebold-Yilmaz volatility / return spillover

Fit a VAR(L) on returns. From the variance decomposition (FEVD) at horizon H, compute:

- **`from(i, j)`** = % of i's H-step forecast variance attributable to shocks in j
- **Net spillover from j to i**: `from(i,j) − from(j,i)` — directional risk transmission
- **Total spillover index**: average of all off-diagonal entries — a single scalar saying how interconnected the group is

In a stat-arb lens, products with large *net-into* spillover are **followers** (absorbing risk from the rest); products with large *net-out* are **leaders**. Trading: short followers vs. long leaders during shocks; reverse in calm. Total index swings indicate regime changes."""))

from statsmodels.tsa.api import VAR

H_FEVD = 10
VAR_L_DY = 4

ret_for_var = rets.dropna()
if len(ret_for_var) > 20000:
    step = max(1, len(ret_for_var) // 20000)
    ret_for_var = ret_for_var.iloc[::step]

try:
    model_dy = VAR(ret_for_var)
    res_dy   = model_dy.fit(VAR_L_DY)
    fevd_arr = res_dy.fevd(H_FEVD).decomp[:, -1, :]   # (N, N): rows i = effect, cols j = source
    fevd_df  = pd.DataFrame(fevd_arr * 100.0, index=columns, columns=columns)
    print(f"=== FEVD at H={H_FEVD} (% of i's forecast variance from j's shocks) ===")
    display(fevd_df.round(2))

    fevd_vals = fevd_df.values
    spillover_from = fevd_df.sum(axis=1) - np.diag(fevd_vals)   # received by i (excl self)
    spillover_to   = fevd_df.sum(axis=0) - np.diag(fevd_vals)   # sent by j (excl self)
    net_spillover  = spillover_to - spillover_from
    total_spillover = (fevd_vals.sum() - np.trace(fevd_vals)) / fevd_vals.sum() * 100

    spill_df = pd.DataFrame({
        "from_others": spillover_from,
        "to_others":   spillover_to,
        "net":         net_spillover,
    }).round(2)
    spill_df.to_csv(OUT_DIR / "dy_spillover.csv")
    print(f"\n=== Spillover summary (total index = {total_spillover:.2f}%) ===")
    display(spill_df)
    print("\n  net > 0  → product transmits risk to the group  (LEADER)")
    print("  net < 0  → product receives risk from the group   (FOLLOWER)")

    fig, axs = plt.subplots(1, 2, figsize=(14, 5.2))
    sns.heatmap(fevd_df, annot=True, fmt=".1f", cmap="rocket_r", ax=axs[0],
                cbar_kws={"label": "% variance"})
    axs[0].set_title(f"FEVD at H={H_FEVD}\nrows = i (effect), cols = j (shock source)")
    axs[0].tick_params(axis="x", labelsize=7, rotation=45)
    axs[0].tick_params(axis="y", labelsize=7)

    spill_df.sort_values("net").plot(kind="barh", ax=axs[1])
    axs[1].set_title("Spillover decomposition per product"); axs[1].axvline(0, color="black", lw=0.7)
    axs[1].tick_params(axis="y", labelsize=7)
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"VAR / FEVD failed: {e}")
    spill_df = pd.DataFrame()
    fevd_df = pd.DataFrame()


In [ ]:
display(Markdown("""## §31 — Principal-component cross-prediction (factor lead-lag)

§8 showed PCA decomposes the group into 5 orthogonal factors. PC1 is the dominant common factor (the "trend"); PC2..PC5 are residual axes.

A subtle multi-product edge: does **PC2(t) predict PC1(t+L)**, or vice versa? The factors are *contemporaneously* orthogonal by construction, but lagged relationships can persist. If a residual factor leads the dominant factor, the residual is an **early-warning** signal for the trend.

We compute `corr(PC_i(t), PC_j(t+L))` for L ∈ {1, 5, 10, 50}, all (i, j) pairs."""))

pca_panel = pd.DataFrame(scores, columns=[f"PC{i+1}" for i in range(P)])
PC_LAGS = [1, 5, 10, 50]

pc_rows = []
for i in range(P):
    for j in range(P):
        if i == j:
            continue
        for L in PC_LAGS:
            x = pca_panel[f"PC{i+1}"]
            y = pca_panel[f"PC{j+1}"].shift(-L)
            m = (~x.isna()) & (~y.isna())
            if m.sum() < 100:
                continue
            c = float(x[m].corr(y[m]))
            pc_rows.append({
                "cause_pc":  f"PC{i+1}",
                "effect_pc": f"PC{j+1}",
                "lag":       L,
                "corr":      c,
                "abs_corr":  abs(c),
            })

pc_pred = pd.DataFrame(pc_rows).sort_values("abs_corr", ascending=False).reset_index(drop=True)
pc_pred.to_csv(OUT_DIR / "pc_cross_prediction.csv", index=False)
print("=== Cross-PC lead-lag (sorted by |corr|; remember contemporaneous corr is exactly 0) ===")
display(pc_pred.head(40).round(5))

# Heatmap per lag
fig, axs = plt.subplots(1, len(PC_LAGS), figsize=(4 * len(PC_LAGS), 4))
for k, L in enumerate(PC_LAGS):
    pivot_lag = pc_pred[pc_pred["lag"] == L].pivot(index="cause_pc", columns="effect_pc", values="corr")
    pivot_lag = pivot_lag.reindex(index=[f"PC{i+1}" for i in range(P)], columns=[f"PC{i+1}" for i in range(P)])
    ax = axs[k] if len(PC_LAGS) > 1 else axs
    sns.heatmap(pivot_lag, annot=True, fmt=".3f", center=0, cmap="coolwarm",
                ax=ax, cbar=False, square=True)
    ax.set_title(f"corr(PC_i(t), PC_j(t+{L}))")
    ax.tick_params(axis="x", labelsize=8); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown("""## §32 — Regime-conditional baskets — recompute by regime

A basket can be stationary in calm regimes and break in vol regimes (or vice versa). We split ticks into regimes by three group-level state variables and recompute spread stationarity within each:

- **Vol regime**: group-mean of rolling `|return|` (top tercile = high vol)
- **Dispersion regime**: cross-sectional std of z-scored mids (high = group fragmenting)
- **|OBI| regime**: average |OBI| across products (high = group is one-sided)

For the §6 best pair AND the §11 best K=2 integer basket we report the spread's ADF p-value inside each regime tercile. A spread that's stationary in low-vol but not high-vol is a **regime-conditional** trade: enter only when the conditioning variable is in the right state."""))

# Group-level regime variables
group_vol_g  = rets.abs().rolling(WINDOW_FAST, min_periods=50).mean().mean(axis=1)
z_mid_g      = (mid_aligned - mid_aligned.mean()) / mid_aligned.std(ddof=0).replace(0, np.nan)
disp_g       = z_mid_g.std(axis=1, ddof=0)
obi_mag_g    = feat_panels["obi"].abs().mean(axis=1)


def regime_split(spread_series, regime_var, name):
    rv = regime_var.reindex(spread_series.index)
    m = (~rv.isna()) & (~spread_series.isna())
    if m.sum() < 200:
        return []
    q33, q67 = np.nanquantile(rv[m], [0.33, 0.67])
    rows = []
    for label, mask in [
        ("low",  (rv <  q33) & m),
        ("mid", ((rv >= q33) & (rv < q67)) & m),
        ("high", (rv >= q67) & m),
    ]:
        sp = spread_series[mask].dropna()
        if len(sp) < 100:
            continue
        rows.append({
            "regime_var":  name,
            "regime":      label,
            "n_ticks":     int(len(sp)),
            "spread_mean": float(sp.mean()),
            "spread_std":  float(sp.std()),
            "adf_p":       adf_p(sp.values),
            "half_life":   half_life(sp.values),
        })
    return rows


regime_rows = []

# §6 best OLS pair
if len(pair_table.dropna(subset=["eg_p"])):
    bp = pair_table.dropna(subset=["eg_p"]).iloc[0]
    a, b, h = bp["a"], bp["b"], bp["hedge_ratio"]
    spread_pair = mid_aligned[a] - h * mid_aligned[b]
    print(f"§6 best pair: {a} − {h:.3f}·{b}")
    for nm, rv in [("vol", group_vol_g), ("dispersion", disp_g), ("|obi|", obi_mag_g)]:
        for r in regime_split(spread_pair, rv, nm):
            r["spread_id"] = "§6_pair"; regime_rows.append(r)

# §11 best K=2 integer basket (if exists)
if 2 in basket_int and len(basket_int[2].dropna(subset=["int_adf_p"])):
    bb = basket_int[2].dropna(subset=["int_adf_p"]).iloc[0]
    target = bb["target"]
    basket = bb["basket"]
    coefs  = np.asarray(bb["int_coefs"], dtype=float)
    S_idx  = [columns.index(p) for p in basket]
    spread_b = (X[:, columns.index(target)] - X[:, S_idx] @ coefs)
    spread_basket = pd.Series(spread_b, index=mid_aligned.index)
    print(f"\n§11 best K=2 integer basket: {bb['int_basket_str']}")
    for nm, rv in [("vol", group_vol_g), ("dispersion", disp_g), ("|obi|", obi_mag_g)]:
        for r in regime_split(spread_basket, rv, nm):
            r["spread_id"] = "§11_K2"; regime_rows.append(r)

# §26 dispersion outlier (if §26 has run, the most stationary one)
if "xs_resid" in dir() and "xs_table" in dir() and len(xs_table.dropna(subset=["naive_resid_adf_p"])):
    rb = xs_table.dropna(subset=["naive_resid_adf_p"]).sort_values("naive_resid_adf_p").iloc[0]
    sp_xs = xs_resid[rb["product"]]
    print(f"\n§26 best naive dispersion residual: {rb['product']} − mean(others)")
    for nm, rv in [("vol", group_vol_g), ("dispersion", disp_g), ("|obi|", obi_mag_g)]:
        for r in regime_split(sp_xs, rv, nm):
            r["spread_id"] = f"§26_{rb['product'][-12:]}"; regime_rows.append(r)

regime_table = pd.DataFrame(regime_rows)
regime_table.to_csv(OUT_DIR / "regime_conditional_basket.csv", index=False)

print("\n=== Regime-conditional ADF on the strongest spreads ===")
display(regime_table.round(4))

# Visualize the regime tercile boundaries on the spread time series for the §6 pair
if "spread_pair" in dir():
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        subplot_titles=["§6 best-pair spread", "group |ret| (vol)",
                                        "cross-sectional dispersion", "group |OBI|"])
    fig.add_trace(go.Scatter(y=spread_pair.values, line=dict(width=0.6), name="spread"), row=1, col=1)
    fig.add_trace(go.Scatter(y=group_vol_g.reindex(spread_pair.index).values, line=dict(width=0.6, color="red"), name="vol"), row=2, col=1)
    fig.add_trace(go.Scatter(y=disp_g.reindex(spread_pair.index).values, line=dict(width=0.6, color="purple"), name="disp"), row=3, col=1)
    fig.add_trace(go.Scatter(y=obi_mag_g.reindex(spread_pair.index).values, line=dict(width=0.6, color="green"), name="|obi|"), row=4, col=1)
    fig.update_layout(height=750, showlegend=False, hovermode="x unified",
                      title=f"{GROUP_LABEL}: spread vs. regime variables")
    fig.show()


In [ ]:
display(Markdown("""## §33 — Trade-flow cross-excitation (proto-Hawkes)

A full Hawkes model is overkill, but we can capture the same idea cheaply. Define **trade arrival rate** `λ_p(t)` = trade count in [t−w, t] per product. Then per ordered pair (a, b):

`corr(λ_a(t), λ_b(t+L))` for L ∈ {0, 1, 5, 10, 20}

If A's recent burst predicts B's near-future burst, we have **cross-excitation** — A's order flow is *triggering* B's. Burst-trading bots often quote-stuff a leader and the followers' bots respond automatically.

We compute:
- **Arrival-rate** correlation (unsigned trade count)
- **Signed-flow** correlation (signed by Lee-Ready from §22)

Big difference between the two = bots react to *any* activity in A but don't (or do) match A's direction."""))

if "flow" in dir() and len(flow) and "trades" in dir() and len(trades):
    trade_count = (trades.groupby(["t", "symbol"]).size()
                   .unstack(fill_value=0).reindex(mid_aligned.index, fill_value=0))
    trade_count = trade_count[[c for c in PRODUCTS if c in trade_count.columns]]

    BURST_WINDOW = 50
    arrival_rate = trade_count.rolling(BURST_WINDOW, min_periods=1).sum()
    signed_rate  = flow.rolling(BURST_WINDOW, min_periods=1).sum()

    print(f"Arrival-rate panel (rolling {BURST_WINDOW}-tick count): shape={arrival_rate.shape}")
    print("Per-product avg arrival rate (trades / window):")
    print(arrival_rate.mean().round(3).to_string())

    LEADS_X = [0, 1, 5, 10, 20]
    excite_rows = []
    for a in arrival_rate.columns:
        for b in arrival_rate.columns:
            row = {"a_burst": a, "b_burst": b}
            for L in LEADS_X:
                xa, xb = arrival_rate[a], arrival_rate[b]
                sa, sb = signed_rate[a],  signed_rate[b]
                if L == 0:
                    row[f"rate_corr_lead_{L}"]   = float(xa.corr(xb))
                    row[f"signed_corr_lead_{L}"] = float(sa.corr(sb))
                else:
                    row[f"rate_corr_lead_{L}"]   = float(xa.corr(xb.shift(-L)))
                    row[f"signed_corr_lead_{L}"] = float(sa.corr(sb.shift(-L)))
            excite_rows.append(row)
    excite_table = pd.DataFrame(excite_rows)
    rate_cols = [f"rate_corr_lead_{L}" for L in LEADS_X]
    sign_cols = [f"signed_corr_lead_{L}" for L in LEADS_X]
    excite_table["max_rate_abs"]   = excite_table[rate_cols].abs().max(axis=1)
    excite_table["max_signed_abs"] = excite_table[sign_cols].abs().max(axis=1)
    excite_table = excite_table.sort_values("max_rate_abs", ascending=False).reset_index(drop=True)
    excite_table.to_csv(OUT_DIR / "trade_excitation.csv", index=False)

    print(f"\n=== Cross-excitation top (window={BURST_WINDOW}, sorted by max |rate-corr|) ===")
    display(excite_table.head(20).round(3))

    print("\n=== Cross-only (a ≠ b) — pure cross-excitation (rates) ===")
    display(excite_table.query("a_burst != b_burst")[
        ["a_burst", "b_burst"] + rate_cols + ["max_rate_abs"]].head(20).round(3))

    print("\n=== Where signed-flow lead is much stronger than arrival-rate lead ===")
    excite_table["sign_minus_rate_gap"] = excite_table["max_signed_abs"] - excite_table["max_rate_abs"]
    display(excite_table.sort_values("sign_minus_rate_gap", ascending=False)[
        ["a_burst", "b_burst", "max_rate_abs", "max_signed_abs", "sign_minus_rate_gap"]].head(15).round(3))
else:
    print("(no trade flow available — §22 must run first)")
    excite_table = pd.DataFrame()


In [ ]:
display(Markdown("""## §34 — Multi-product niche dashboard + how to read

Roll up the strongest **multi-product** findings from §25-§33 — the strategies that genuinely use the whole 5-product system, not just pairs."""))

mp_niche = []

# §25 k-of-N tail confluence
if "k_table" in dir() and len(k_table):
    high_k = k_table.query("k >= 2 and lift > 1.0").copy() if "lift" in k_table.columns else pd.DataFrame()
    if len(high_k):
        r = high_k.sort_values("lift", ascending=False).iloc[0]
        mp_niche.append({"section": "§25 k-of-N tail confluence",
                         "what": f"k={int(r['k'])} simultaneous tail products",
                         "score": float(r["lift"]),
                         "metric": f"obs={int(r['n_obs'])}, exp={r['n_exp']:.2f}, z={r['z_score']:.1f}"})

# §26 dispersion outlier
if "xs_table" in dir() and len(xs_table):
    rok = xs_table.dropna(subset=["r_xs_adf_p"])
    if len(rok):
        r = rok.iloc[0]
        mp_niche.append({"section": "§26 dispersion outlier",
                         "what": r["product"],
                         "score": -np.log10(max(r["r_xs_adf_p"], 1e-300)),
                         "metric": f"ADF p={r['r_xs_adf_p']:.3g}, half-life={r['r_xs_half_life']}"})

# §27 Johansen
if "jbasket_table" in dir() and len(jbasket_table):
    rok = jbasket_table.dropna(subset=["adf_p"]).sort_values("adf_p")
    if len(rok):
        r = rok.iloc[0]
        weights_str = " + ".join(f"{w:+.2f}·{p[-12:]}" for p, w in r['weights'].items() if abs(w) > 0.01)
        mp_niche.append({"section": "§27 Johansen basket",
                         "what": weights_str,
                         "score": -np.log10(max(r["adf_p"], 1e-300)),
                         "metric": f"ADF p={r['adf_p']:.3g}, half-life={r['half_life']}"})

# §28 Lasso-VAR
if "network_table" in dir() and len(network_table):
    r = network_table.iloc[0]
    mp_niche.append({"section": "§28 Lasso-VAR top edge",
                     "what": f"{r['cause']} → {r['effect']} (lag {int(r['lag'])})",
                     "score": float(r["abs_coef"]),
                     "metric": f"coef={r['coef']:.4f}"})

# §29 partial-corr MST
if "mst_df" in dir() and len(mst_df):
    rs = mst_df.reindex(mst_df["partial_corr"].abs().sort_values(ascending=False).index)
    r = rs.iloc[0]
    mp_niche.append({"section": "§29 partial-corr MST top",
                     "what": f"{r['a']} ↔ {r['b']}",
                     "score": float(abs(r["partial_corr"])),
                     "metric": f"partial_corr={r['partial_corr']:.3f}"})

# §30 spillover
if "spill_df" in dir() and len(spill_df):
    leader = spill_df.sort_values("net", ascending=False).iloc[0]
    follower = spill_df.sort_values("net").iloc[0]
    mp_niche.append({"section": "§30 top net leader",   "what": leader.name,
                     "score": float(leader["net"]),
                     "metric": f"to={leader['to_others']:.1f}%, from={leader['from_others']:.1f}%"})
    mp_niche.append({"section": "§30 top net follower", "what": follower.name,
                     "score": float(-follower["net"]),
                     "metric": f"to={follower['to_others']:.1f}%, from={follower['from_others']:.1f}%"})

# §31 PC cross
if "pc_pred" in dir() and len(pc_pred):
    r = pc_pred.iloc[0]
    mp_niche.append({"section": "§31 PC cross-lead",
                     "what": f"{r['cause_pc']} → {r['effect_pc']} (lag {int(r['lag'])})",
                     "score": float(r["abs_corr"]),
                     "metric": f"corr={r['corr']:.4f}"})

# §32 regime
if "regime_table" in dir() and len(regime_table):
    rok = regime_table.dropna(subset=["adf_p"]).sort_values("adf_p")
    if len(rok):
        r = rok.iloc[0]
        mp_niche.append({"section": "§32 regime-best basket",
                         "what": f"{r['regime_var']}={r['regime']}",
                         "score": -np.log10(max(r["adf_p"], 1e-300)),
                         "metric": f"ADF p={r['adf_p']:.3g}, n={int(r['n_ticks'])}, hl={r['half_life']}"})

# §33 cross-excitation
if "excite_table" in dir() and len(excite_table):
    cross = excite_table.query("a_burst != b_burst")
    if len(cross):
        r = cross.iloc[0]
        mp_niche.append({"section": "§33 trade-burst lead",
                         "what": f"{r['a_burst']} → {r['b_burst']}",
                         "score": float(r["max_rate_abs"]),
                         "metric": f"max rate-corr={r['max_rate_abs']:.3f}"})

mp_niche_df = pd.DataFrame(mp_niche)
mp_niche_df.to_csv(OUT_DIR / "multi_product_niche_dashboard.csv", index=False)
display(Markdown("### Multi-product niche dashboard — strongest §25-§33 findings"))
display(mp_niche_df.round(4))

display(Markdown("""
### Reading these multi-product signals

| Signal | What it gives you | Trading lens |
|---|---|---|
| **§25 k-of-N tail** | Empirical distribution of "how many products are extreme at the same tick" vs. binomial null. Lift at high k = group shock channel | When ≥3 products are simultaneously extreme, position-size *down* on baskets — your "spread" probably broke. Use k as a regime gate |
| **§26 dispersion outlier** | Per-product cross-sectional z-position. Stationary → outliers mean-revert toward the group | Short the outlier vs. equal-weight basket of the others. Built-in integer weights, naturally hedged |
| **§27 Johansen** | Number of independent stationary baskets in the 5-product system + eigenvector basket weights | Trade *each* cointegrating eigenvector as an independent spread (orthogonal P&L by construction) |
| **§28 Lasso-VAR** | Sparse directed lead-lag network. Edges that survive Lasso are *direct*, not mediated | Use as the conditional set when constructing alpha — only add features that survive; combine forecasts at lag 1 |
| **§29 Partial-corr MST** | Skeleton of direct dependencies. Hub = group's information carrier; leaves = followers | Trade leaves vs. hub: their idiosyncratic deviation is most likely to revert |
| **§30 DY spillover** | Per-product %-contribution to others' forecast variance. Net leader vs. net follower | Long leaders / short followers during shocks; reverse in calm. Total spillover index is a regime indicator |
| **§31 PC cross-lead** | PCs are contemporaneously orthogonal but can lead each other | If PC2 leads PC1, PC2 is an early-warning signal for the trend factor — trade with PC2's sign |
| **§32 Regime baskets** | Stationarity of the §6 best pair conditional on group-vol / dispersion / OBI regime | Only enter the pair-trade in the regime where it's stationary. Halve size in border regimes |
| **§33 Cross-excitation** | Lagged trade-burst correlation. A's burst → B's near-future burst = bot mirroring | Trigger an immediate B trade off A's burst; expect tighter window than §22 flow |

### Strategy ladder — multi-product (least to most ambitious)

1. **Outlier reversion (§26)** — single signal, single trade per outlier. Easiest to deploy, integer weights, no hedge ratio fitting needed.
2. **Cointegrating eigenvectors (§27)** — multiple orthogonal spreads at once. Requires Johansen rank ≥ 2; allocate independent capital per eigenvector.
3. **Regime-gated baskets (§32)** — gate the §11 integer baskets by §30/§32 regime variables. Cuts drawdowns from regime breaks.
4. **Lead-lag bot (§28 + §31 + §33)** — react within ticks to lagged predictions from the directed network. Requires bot-class infrastructure.
5. **Group-shock playbook (§25 + §30)** — when k-of-N tail or DY spillover spikes, deleverage and reverse. Macro-state overlay on top of every spread.
"""))
